# Modul 1 · Git, Tests und Teamarbeit
**Advanced Data Science · Advanced AI Engineering** · linkit courses

Am 1. Oktober 2026 kommt die Nachlieferung für den 28. bis 30. September, und mit ihr zwei Meldungen. Der Importer aus Modul 0 verwirft alle zwölf Zeilen aus Illinois (Issue 1, das löst du auf deinem Laptop). Und die Werkleitung Illinois schreibt zum Wochenbericht, den Lena Brandt aus dem Controlling des Stammwerks gebaut hat: „The weekly report flags week 35 for high scrap, but our scrap peak was in week 36. Week 27 looks too low, too.“ Lenas Modul `kennzahlen.py` hat keinen einzigen Test.

In diesem Notebook baust du zuerst die Grundideen von Git in Python nach: Inhalte mit IDs, eine Historie mit Branches, die Merge-Basis, „vorne“ und „hinten“ und den Dreiwege-Merge. Danach schreibst du echte Tests mit pytest für Lenas Modul, findest damit die Fehler, behebst sie und beantwortest die Meldung aus Illinois mit einem Integrationstest an den echten Daten. Modul und Tests wandern am Ende in dein Repository.

**Leitfrage:** Wie arbeiten zwei Personen am selben Code, ohne dass etwas verloren geht oder unbemerkt kaputtgeht?

**Lernziele**
1. Arbeitsverzeichnis, Staging-Bereich und Repository sicher unterscheiden
2. Mit Branches arbeiten, mergen und Konflikte lösen
3. Über GitHub mit Issues, Pull Requests und Reviews zusammenarbeiten
4. Tests mit pytest schreiben, lesen und gezielt auswählen
5. Fixtures, kleine Testdaten und Regressionstests einsetzen
6. pre-commit und CI einrichten und rote Läufe lesen
7. Versionen vergeben, Releases anlegen und sicher rückgängig machen

**So arbeitest du**
- Das Notebook läuft auf der Kursseite direkt im Browser, mit echtem pytest. Der Hauptweg ist dein Laptop: Lade das Notebook herunter, leg es in deinem Projekt unter `notebooks/` ab und öffne es in VS Code mit dem Kernel aus deiner `.venv`. pytest kommt auf dem Laptop erst im Schritt L4 ins Projekt; öffnest du das Notebook vorher in VS Code, führ im Projektordner zuerst `uv add --dev pytest` aus. Colab geht auch, ist aber nicht nötig.
- Das Notebook arbeitet in seinem eigenen Ordner, die Daten liegen deshalb unter `daten/` neben dem Notebook. Alle Pfade sind relativ.
- Führe die Zellen von oben nach unten mit **Shift + Enter** aus.
- In Teil A schreibst du Funktionen: Wo `raise NotImplementedError("TODO 2")` steht, kommt dein Code hin. Ab Teil B schreibst du Tests in Zellen, die mit `# → tests/test_kennzahlen.py` beginnen. Wo `...  # TODO 5` steht, gehört ein Test hin.
- Unter jeder Aufgabe prüft ein Selbsttest dein Ergebnis: ✅ stimmt, ❌ stimmt noch nicht (mit Hinweis), ⏳ wartet auf ein früheres TODO. Das Notebook läuft in jedem Fall weiter, auch mit „Alle ausführen“.
- Mindestziel sind TODO 1 bis 12. Die beiden Bonusaufgaben vertiefen.
- Jeder Teil gehört zu einem Abschnitt des Skripts. Hilfen zu jedem TODO und typische Fehler findest du auf der Kursseite.

In [ ]:
# Hilfen für die Selbsttests: einfach ausführen, nicht ändern.
import contextlib
import datetime as dt
import difflib
import hashlib
import inspect
import io
import json
import logging
import os
import re
from pathlib import Path


GRUEN = {}


def check(name, test):
    """Führt einen Test aus: ✅ stimmt, ❌ stimmt noch nicht, ⏳ ein früheres TODO ist noch offen."""
    eigenes = re.match(r"TODO \d+|Bonus \d", name).group(0)
    GRUEN[name] = False
    try:
        ergebnis = test()
    except NotImplementedError as e:
        if str(e) and str(e) != eigenes:
            print(f"⏳ {name}: wartet auf {e}")
        else:
            print(f"❌ {name}: noch nicht gelöst")
    except AssertionError as e:
        print(f"❌ {name}: {e}")
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}")
    else:
        GRUEN[name] = True
        print(f"✅ {name}" + (f": {ergebnis}" if isinstance(ergebnis, str) else ""))


def braucht(*labels):
    """Ein Test, der frühere TODOs nutzt, wartet, bis deren Selbsttests alle ✅ zeigen."""
    for label in labels:
        eigene = [ok for name, ok in GRUEN.items() if re.match(r"TODO \d+|Bonus \d", name).group(0) == label]
        if not eigene or not all(eigene):
            raise NotImplementedError(label)


def zeige(was):
    """Zeigt ein Zwischenergebnis. Solange ein nötiges TODO offen ist, steht dort nur ein Hinweis."""
    try:
        wert = was()
    except NotImplementedError as e:
        print(f"(noch offen: {e})")
    except Exception as e:
        print(f"(Fehler: {type(e).__name__}: {e})")
    else:
        if wert is not None:
            print(wert)


@contextlib.contextmanager
def ruhig():
    """Während des Blocks schreiben die Logger unter "nordwerk" nichts."""
    logger = logging.getLogger("nordwerk")
    vorher = logger.level
    logger.setLevel(logging.CRITICAL + 10)
    try:
        yield
    finally:
        logger.setLevel(vorher)


print("Bereit.")

In [ ]:
# Daten laden: Auf der Kursseite liegt die Datei schon bereit, in VS Code und Colab holt diese Zelle sie von GitHub.
import urllib.request

DATEN = ["schichten.csv"]
QUELLE = "https://raw.githubusercontent.com/juliankunz18/linkit-courses/main/advanced/daten/"

# Alle Pfade sind relativ zum Ordner dieses Notebooks: in VS Code also notebooks/daten/, im Browser daten/.
DATENORDNER = Path("daten")
DATENORDNER.mkdir(exist_ok=True)
for datei in DATEN:
    ziel = DATENORDNER / datei
    if not ziel.exists():
        teil = ziel.with_name(ziel.name + ".teil")  # erst vollständig laden, dann umbenennen
        urllib.request.urlretrieve(QUELLE + datei, teil)
        teil.replace(ziel)
SCHICHTEN_CSV = (DATENORDNER / "schichten.csv").resolve()
_kopf = SCHICHTEN_CSV.read_text(encoding="utf-8").split("\n", 1)[0]
if not _kopf.startswith("werk,datum,linie,schicht,"):
    SCHICHTEN_CSV.unlink()
    raise RuntimeError("schichten.csv war beschädigt und ist gelöscht. Führ diese Zelle noch einmal aus.")
print("schichten.csv bereit:", len(SCHICHTEN_CSV.read_text(encoding="utf-8").splitlines()) - 1, "Zeilen in", DATENORDNER / "schichten.csv")

## A · Git ohne Git
*Lerneinheiten 1 und 2 · Skript Abschnitte 1.2 und 2.3 bis 2.5*

Git ist kein Zauberwerk. Es speichert Inhalte unter einer ID, die aus dem Inhalt selbst berechnet ist, und merkt sich für jeden Commit, auf welchen Commits er aufbaut. Fast alles andere folgt daraus: was ein Branch ist, was „ahead“ heißt und wie ein Merge zwei Änderungen zusammenbringt. In diesem Teil baust du genau das in Python nach. Das Skript zeigt dieselben Ideen mit echten Git-Befehlen an einem Probe-Repository.

### TODO 1 · Git-ID eines Inhalts
Git speichert jede Datei als **Blob** (binary large object). Die ID eines Blobs ist der SHA-1-Hash von einem kurzen Kopf und dem Inhalt. Deshalb hat derselbe Inhalt auf jedem Rechner dieselbe ID, und jede noch so kleine Änderung ergibt eine andere.

**Ziel:** `blob_id(daten)` berechnet die ID, die `git hash-object` für eine Datei mit dem Inhalt `daten` ausgibt.

**Signatur:** `blob_id(daten: bytes) -> str`

**Regeln**
- Der Kopf besteht aus dem Wort `blob`, einem Leerzeichen, der Länge des Inhalts in Byte als Dezimalzahl und einem Nullbyte: für `b"hello\n"` also `b"blob 6\0"`.
- Gehasht wird Kopf plus Inhalt mit `hashlib.sha1(…)`. Das Ergebnis ist `hexdigest()`, 40 Zeichen.

In [ ]:
def blob_id(daten: bytes) -> str:
    """Die Git-ID eines Blobs: SHA-1 von b"blob <Länge>\0" plus Inhalt, als 40 Hexadezimalzeichen."""
    raise NotImplementedError("TODO 1")

In [ ]:
# Selbsttest TODO 1
NORDWERK_TOML = b'[import]\neingang = "daten/roh"\nausgang = "daten/aufbereitet"\nlog_level = "WARNING"\n'
zeige(lambda: {"leer": blob_id(b"")[:7], "hello": blob_id(b"hello\n")[:7], "nordwerk.toml": blob_id(NORDWERK_TOML)[:7]})


def _t1():
    faelle = [("die leere Datei", b"", "e69de29bb2d1d6434b8b29ae775ad8c2e48c5391"),
              ("b'hello\\n'", b"hello\n", "ce013625030ba8dba906f756967f9e9ca394464a"),
              ("nordwerk.toml aus Modul 0", NORDWERK_TOML, "5c6c2e7b5ecbf3c372188fc5342d55f47b5a3092")]
    for name, daten, soll in faelle:
        ist = blob_id(daten)
        assert isinstance(ist, str) and len(ist) == 40, f"blob_id liefert {ist!r}, erwartet 40 Hexadezimalzeichen"
        if ist == hashlib.sha1(daten).hexdigest():
            raise AssertionError(f"{name}: nur der Inhalt gehasht, der Kopf b'blob <Länge>\\0' fehlt")
        if ist == hashlib.sha1(b"blob \0" + daten).hexdigest():
            raise AssertionError(f"{name}: im Kopf fehlt die Länge des Inhalts")
        if ist == hashlib.sha1(f"blob {len(daten)}".encode() + daten).hexdigest():
            raise AssertionError(f"{name}: im Kopf fehlt das Nullbyte nach der Länge")
        assert ist == soll, f"{name}: {ist[:7]}…, erwartet {soll[:7]}… (wie git hash-object)"


check("TODO 1 drei Inhalte wie git hash-object", _t1)

Die leere Datei hat in jedem Git-Repository der Welt die ID `e69de29`. `nordwerk.toml` aus Modul 0 hat `5c6c2e7`, solange die Zeilen mit `\n` enden. Mit Windows-Zeilenenden `\r\n` wäre es ein anderer Inhalt und damit eine andere ID: Probier `blob_id(NORDWERK_TOML.replace(b"\n", b"\r\n"))` aus. Weil die ID aus dem Inhalt folgt, merkt Git jede Änderung, auch eine unsichtbare.

### Die Historie eines Teams
Ein Commit speichert einen Schnappschuss aller Dateien, die Nachricht und die IDs seiner **Eltern**: einen bei einem normalen Commit, zwei bei einem Merge-Commit, keinen beim ersten Commit. Ein **Branch** ist nur ein Zeiger auf einen Commit. Die nächste Zelle gibt dir die Historie eines Repositorys, an dem du mit Kim arbeitest, als Wörterbuch: Commit-Kürzel (die ersten sieben Zeichen der ID) → Liste der Eltern. Die Historie stammt aus einem echten Repository. So zeigt `git log --oneline --graph --all` sie:

```
* d754b36 (origin/main) CI: Matrix für Windows und macOS
* 4beef24 Datenblatt verlinkt
| * c83c7db (HEAD -> main) Version 0.2.0
|/
*   5df95c7 Merge branch 'fix/illinois-schicht'
|\
| * 53bb8fd (fix/illinois-schicht) Illinois: neue Schichtnamen erkennen
* | 87635aa CI: Lieferungen vor den Tests laden
* | 1e5bac2 README: Nachlieferung beschrieben
| | * 235946c (feature/kennzahlen) Tests für die Kennzahlen
| | * 5f2f33b Kennzahlen der Kollegin übernommen
| |/
|/|
* | 217b739 pre-commit und CI eingerichtet
|/
* ec39b49 Vertragstests mit pytest
* 299e1d0 (tag: v0.1.0) Importer aus Modul 0
```

Lies den Graphen von unten nach oben. Der Branch `fix/illinois-schicht` ist früh von `ec39b49` abgezweigt und mit `5df95c7` in `main` zurückgeflossen. `feature/kennzahlen` zweigt bei `217b739` ab. Kim hat zwei Commits nach GitHub geschickt (`origin/main`), die dein `main` noch nicht kennt, und dein Commit `c83c7db` liegt noch nicht auf GitHub.

In [ ]:
# Vorgabe: die Historie als Wörterbuch (Kürzel → Eltern, beim Merge-Commit zuerst der Branch, in den gemergt wurde)
HISTORIE = {
    "299e1d0": [],
    "ec39b49": ["299e1d0"],
    "53bb8fd": ["ec39b49"],
    "217b739": ["ec39b49"],
    "1e5bac2": ["217b739"],
    "87635aa": ["1e5bac2"],
    "5df95c7": ["87635aa", "53bb8fd"],
    "5f2f33b": ["217b739"],
    "235946c": ["5f2f33b"],
    "c83c7db": ["5df95c7"],
    "4beef24": ["5df95c7"],
    "d754b36": ["4beef24"],
}
NACHRICHTEN = {
    "299e1d0": "Importer aus Modul 0",
    "ec39b49": "Vertragstests mit pytest",
    "53bb8fd": "Illinois: neue Schichtnamen erkennen\n\nDas Berichtswerkzeug schreibt seit dem 28. September 1st Shift und\n2nd Shift statt 1st und 2nd. Fixes #1",
    "217b739": "pre-commit und CI eingerichtet",
    "1e5bac2": "README: Nachlieferung beschrieben",
    "87635aa": "CI: Lieferungen vor den Tests laden",
    "5df95c7": "Merge branch 'fix/illinois-schicht'",
    "5f2f33b": "Kennzahlen der Kollegin übernommen",
    "235946c": "Tests für die Kennzahlen",
    "c83c7db": "Version 0.2.0",
    "4beef24": "Datenblatt verlinkt",
    "d754b36": "CI: Matrix für Windows und macOS",
}
ZEIGER = {"main": "c83c7db", "origin/main": "d754b36", "fix/illinois-schicht": "53bb8fd", "feature/kennzahlen": "235946c"}

for kuerzel, eltern in reversed(HISTORIE.items()):
    namen = [name for name, ziel in ZEIGER.items() if ziel == kuerzel]
    print(kuerzel, f"Eltern {eltern}".ljust(32), NACHRICHTEN[kuerzel].split("\n")[0], f"({', '.join(namen)})" if namen else "")

### TODO 2 · Vorfahren und Merge-Basis
Will Git zwei Branches zusammenführen, sucht es zuerst die **Merge-Basis**: den letzten Stand, den beide gemeinsam haben. Das Skript zeigt sie mit `git merge-base` und als Abbildung (Abschnitt 2.4). Hier berechnest du sie selbst.

**Ziel:** `vorfahren(historie, commit)` liefert alle Commits, die in der Geschichte von `commit` liegen, `merge_basis(historie, a, b)` den nächsten gemeinsamen Vorfahren von `a` und `b`.

**Signaturen:** `vorfahren(historie: dict[str, list[str]], commit: str) -> set[str]` und `merge_basis(historie: dict[str, list[str]], a: str, b: str) -> str`

**Regeln**
- `vorfahren` enthält `commit` selbst, seine Eltern, deren Eltern und so weiter, über **alle** Eltern eines Merge-Commits. So zählt auch `git rev-list`. Ein unbekanntes Kürzel löst `KeyError` aus.
- Gemeinsame Vorfahren sind die Commits, die in beiden Mengen liegen. Davon ist die Merge-Basis derjenige, der **kein Vorfahr eines anderen gemeinsamen Vorfahren** ist: der jüngste gemeinsame Stand.
- Vorsicht mit Abkürzungen: Der gemeinsame Vorfahr mit den wenigsten Schritten ist nicht immer der jüngste. Bei Merge-Commits führt manchmal ein kurzer Weg zu einem alten Commit.
- Gibt es mehrere solche Commits (das passiert nur, wenn Branches über Kreuz gemergt wurden, in dieser Historie nicht), löse `ValueError` aus.

In [ ]:
def vorfahren(historie: dict[str, list[str]], commit: str) -> set[str]:
    """commit und alle Commits, auf denen er aufbaut (über alle Eltern)."""
    raise NotImplementedError("TODO 2")


def merge_basis(historie: dict[str, list[str]], a: str, b: str) -> str:
    """Der nächste gemeinsame Vorfahr von a und b, wie git merge-base."""
    raise NotImplementedError("TODO 2")

In [ ]:
# Selbsttest TODO 2
zeige(lambda: f"Merge-Basis von main und feature/kennzahlen: {merge_basis(HISTORIE, ZEIGER['main'], ZEIGER['feature/kennzahlen'])}")


def _t2_vorfahren():
    v = vorfahren(HISTORIE, ZEIGER["main"])
    assert isinstance(v, set), f"vorfahren liefert {type(v).__name__}, erwartet eine Menge (set)"
    assert ZEIGER["main"] in v, "der Commit selbst gehört zu seinen Vorfahren (so zählt auch git rev-list)"
    soll = {"c83c7db", "5df95c7", "87635aa", "1e5bac2", "217b739", "53bb8fd", "ec39b49", "299e1d0"}
    assert "53bb8fd" in v, "der zweite Elternteil des Merge-Commits 5df95c7 fehlt: Folge allen Eltern"
    assert v == soll, f"vorfahren(main) hat {len(v)} Commits, erwartet 8: zu viel {sorted(v - soll)}, fehlt {sorted(soll - v)}"
    assert vorfahren(HISTORIE, "299e1d0") == {"299e1d0"}, "der erste Commit hat nur sich selbst als Vorfahr"
    assert len(vorfahren(HISTORIE, ZEIGER["origin/main"])) == 9, "vorfahren(origin/main) sollte 9 Commits haben"
    try:
        vorfahren(HISTORIE, "0000000")
    except KeyError:
        pass
    else:
        raise AssertionError("ein unbekanntes Kürzel soll KeyError auslösen")


def _t2_basis():
    m, o, f, k = ZEIGER["main"], ZEIGER["origin/main"], ZEIGER["fix/illinois-schicht"], ZEIGER["feature/kennzahlen"]
    faelle = [("main", "feature/kennzahlen", m, k, "217b739"), ("feature/kennzahlen", "main", k, m, "217b739"),
              ("main", "origin/main", m, o, "5df95c7"), ("fix/illinois-schicht", "main", f, m, "53bb8fd"),
              ("53bb8fd", "217b739", "53bb8fd", "217b739", "ec39b49"), ("origin/main", "feature/kennzahlen", o, k, "217b739"),
              ("299e1d0", "299e1d0", "299e1d0", "299e1d0", "299e1d0")]
    for na, nb, a, b, soll in faelle:
        ist = merge_basis(HISTORIE, a, b)
        if ist != soll and ist in vorfahren(HISTORIE, soll):
            raise AssertionError(f"merge_basis({na}, {nb}) ist {ist}: ein gemeinsamer Vorfahr, aber nicht der jüngste ({soll} baut auf ihm auf)")
        assert ist == soll, f"merge_basis({na}, {nb}) ist {ist!r}, erwartet {soll} (wie git merge-base)"


def _t2_klein():
    raute = {"1": [], "2": ["1"], "3": ["1"], "4": ["2", "3"], "5": ["3"]}
    assert merge_basis(raute, "4", "5") == "3", f"kleine Historie: merge_basis('4', '5') ist {merge_basis(raute, '4', '5')!r}, erwartet '3'"
    kreuz = {"1": [], "2": ["1"], "3": ["1"], "4": ["2", "3"], "5": ["3", "2"]}
    try:
        ergebnis = merge_basis(kreuz, "4", "5")
    except ValueError:
        pass
    else:
        raise AssertionError(f"über Kreuz gemergt: 2 und 3 sind beide beste gemeinsame Vorfahren, erwartet ValueError statt {ergebnis!r}")


check("TODO 2 Vorfahren", _t2_vorfahren)
check("TODO 2 Merge-Basis wie git merge-base", _t2_basis)
check("TODO 2 kleine Historien", _t2_klein)

Die Merge-Basis von `main` und `feature/kennzahlen` ist `217b739`, obwohl `ec39b49` von `main` aus über den Fix-Branch in weniger Schritten erreichbar ist. Mit `ec39b49` als Basis würde ein Merge die Änderungen aus `217b739` noch einmal als neu ansehen. Liegt einer der beiden Commits in der Geschichte des anderen, ist er selbst die Basis: Dann kann Git einfach den Zeiger vorschieben (Fast-Forward), wie bei `fix/illinois-schicht` und `main`.

### TODO 3 · Vorne und hinten
`git status` sagt dir, ob dein Branch seinem Gegenstück auf GitHub voraus ist („ahead“), hinterherhinkt („behind“) oder beides. Das Skript zeigt das an einem zweiten Klon (Abschnitt 3.3), `git rev-list --left-right --count main...origin/main` liefert die beiden Zahlen.

**Ziel:** `vorne_hinten(historie, lokal, entfernt)` zählt die Commits, die nur `lokal` hat, und die, die nur `entfernt` hat.

**Signatur:** `vorne_hinten(historie: dict[str, list[str]], lokal: str, entfernt: str) -> tuple[int, int]`, Rückgabe `(vorne, hinten)`

**Regeln**
- `vorne`: Commits in den Vorfahren von `lokal`, die nicht in den Vorfahren von `entfernt` liegen. Die schickt `git push` hoch.
- `hinten`: umgekehrt. Die holt `git pull`.
- Nutz `vorfahren` aus TODO 2.

In [ ]:
def vorne_hinten(historie: dict[str, list[str]], lokal: str, entfernt: str) -> tuple[int, int]:
    """(Commits nur in lokal, Commits nur in entfernt), wie git rev-list --left-right --count lokal...entfernt."""
    raise NotImplementedError("TODO 3")

In [ ]:
# Selbsttest TODO 3
zeige(lambda: f"main gegen origin/main: {vorne_hinten(HISTORIE, ZEIGER['main'], ZEIGER['origin/main'])}")


def _t3():
    braucht("TODO 2")
    faelle = [("main", "origin/main", (1, 2)), ("feature/kennzahlen", "main", (2, 5)), ("fix/illinois-schicht", "main", (0, 5)),
              ("origin/main", "feature/kennzahlen", (6, 2)), ("main", "main", (0, 0))]
    for lokal, entfernt, soll in faelle:
        ist = vorne_hinten(HISTORIE, ZEIGER[lokal], ZEIGER[entfernt])
        assert isinstance(ist, tuple) and len(ist) == 2, f"vorne_hinten liefert {ist!r}, erwartet ein Paar (vorne, hinten)"
        if ist == soll[::-1] and soll[0] != soll[1]:
            raise AssertionError(f"{lokal} gegen {entfernt}: {ist}, vorne und hinten vertauscht (erwartet {soll})")
        assert ist == soll, f"{lokal} gegen {entfernt}: {ist}, erwartet {soll} (wie git rev-list --left-right --count)"


check("TODO 3 vorne und hinten wie git rev-list", _t3)

Dein `main` ist `origin/main` einen Commit voraus und zwei hinterher: Ihr habt beide weitergearbeitet, die Branches sind auseinandergelaufen. Ein `git push` wird jetzt abgelehnt, bis du Kims zwei Commits mit `git pull` geholt hast. `fix/illinois-schicht` ist `main` nur hinterher; ein Merge von `main` in diesen Branch wäre ein Fast-Forward.

### TODO 4 · Dreiwege-Merge
Für einen Merge vergleicht Git beide Seiten mit der Merge-Basis: Was hat **unsere** Seite geändert, was **ihre**? Änderungen an verschiedenen Stellen übernimmt es beide. Ändern beide Seiten dieselbe Stelle verschieden, entsteht ein **Konflikt** mit Markern, die du in VS Code auflöst (Skript Abschnitt 2.5).

Die vorgegebene Funktion `aenderungen(basis, neu)` findet die geänderten Bereiche einer Seite mit `difflib.SequenceMatcher` aus der Standardbibliothek. Jeder Bereich ist ein Tripel `(start, ende, zeilen)`: Die Zeilen `basis[start:ende]` werden durch `zeilen` ersetzt. `start` und `ende` zählen immer in der Basis; ist `start == ende`, wird dort nur etwas eingefügt.

In [ ]:
# Vorgabe für TODO 4: geänderte Bereiche einer Seite gegenüber der Basis
def aenderungen(basis: list[str], neu: list[str]) -> list[tuple[int, int, list[str]]]:
    """Die Bereiche, in denen neu von basis abweicht: (start, ende, neue_zeilen), start und ende zählen in basis."""
    vergleich = difflib.SequenceMatcher(None, basis, neu, autojunk=False)
    return [(i1, i2, neu[j1:j2]) for art, i1, i2, j1, j2 in vergleich.get_opcodes() if art != "equal"]


BASIS_README = ["# nordwerk-analysen", "Importer für die Schichtmeldungen beider Werke.", "## Aufruf", "uv run nordwerk import",
                "## Daten", "Die Lieferungen liegen in daten/roh.", "## Tests", "uv run pytest"]
print(aenderungen(BASIS_README, BASIS_README[:4] + BASIS_README[6:] + ["uv run pytest -m daten"]))

Das Beispiel löscht die beiden Zeilen zu den Daten (`(4, 6, [])`) und hängt am Ende eine Zeile an (`(8, 8, […])`).

**Ziel:** `drei_wege(basis, unsere, ihre)` führt beide Seiten zusammen und zählt die Konflikte.

**Signatur:** `drei_wege(basis: list[str], unsere: list[str], ihre: list[str]) -> tuple[list[str], int]`; alle Texte sind Listen von Zeilen ohne Zeilenende.

**Regeln**
1. Hol die Bereiche beider Seiten mit `aenderungen` und sortiere sie zusammen nach `start` (bei gleichem Start nach `ende`).
2. Fasse Bereiche zu einem **Block** zusammen, solange der nächste Bereich beginnt, bevor der Block endet, oder genau dort (`start <= ende` des Blocks). So behandelt auch Git direkt benachbarte Änderungen: als eine Stelle.
3. Zwischen den Blöcken übernimmst du die Zeilen der Basis.
4. Für jeden Block baust du aus der Basis den Text **jeder Seite** über die ganze Breite des Blocks (die Bereiche dieser Seite eingesetzt, sonst die Basis). Hat nur eine Seite dort etwas geändert oder sind beide Texte gleich, übernimm diesen Text. Sonst ist es ein Konflikt: `<<<<<<< unsere`, die Zeilen unserer Seite, `=======`, die Zeilen ihrer Seite, `>>>>>>> ihre`.
5. Rückgabe: die Zeilen des Ergebnisses und die Zahl der Konflikte, wie `git merge-file -p -L unsere -L basis -L ihre` und dessen Exit-Code.

In [ ]:
def drei_wege(basis: list[str], unsere: list[str], ihre: list[str]) -> tuple[list[str], int]:
    """Führt unsere und ihre Änderungen gegenüber basis zusammen; liefert (zeilen, konflikte)."""
    raise NotImplementedError("TODO 4")

In [ ]:
# Selbsttest TODO 4
def _mit(**ersetzt):
    """BASIS_README mit ersetzten Zeilen, etwa _mit(z3="…") für die Zeile mit Index 3."""
    zeilen = list(BASIS_README)
    for name, text in ersetzt.items():
        zeilen[int(name[1:])] = text
    return zeilen


def _k(unsere, ihre):
    return ["<<<<<<< unsere", *unsere, "=======", *ihre, ">>>>>>> ihre"]


B = BASIS_README
A1, A3, A5, A7 = "Importer und Kennzahlen für beide Werke.", "uv run nordwerk import daten/roh", "Die Lieferungen liegen in daten/roh, die Nachlieferung in daten/nachlieferung.", "uv run pytest -ra"
FAELLE_MERGE = [
    ("nur unsere Seite", _mit(z3=A3), B, _mit(z3=A3), 0),
    ("nur ihre Seite", B, _mit(z7=A7), _mit(z7=A7), 0),
    ("verschiedene Stellen", _mit(z1=A1), _mit(z5=A5), _mit(z1=A1, z5=A5), 0),
    ("gleiche Änderung auf beiden Seiten", _mit(z7=A7), _mit(z7=A7), _mit(z7=A7), 0),
    ("dieselbe Zeile verschieden", _mit(z1=A1), _mit(z1="Importer für Stammwerk und Illinois."),
     [B[0], *_k([A1], ["Importer für Stammwerk und Illinois."]), *B[2:]], 1),
    ("benachbarte Zeilen", _mit(z2="## Aufruf auf dem Laptop"), _mit(z3="uv run nordwerk import --log-level INFO"),
     [*B[:2], *_k(["## Aufruf auf dem Laptop", B[3]], [B[2], "uv run nordwerk import --log-level INFO"]), *B[4:]], 1),
    ("Einfügen und Löschen", [*B, "uv run pytest -m daten"], B[:4] + B[6:], [*B[:4], *B[6:], "uv run pytest -m daten"], 0),
    ("beide hängen etwas an", [*B, "## Version", "0.2.0"], [*B, "## Lizenz", "nur intern"], [*B, *_k(["## Version", "0.2.0"], ["## Lizenz", "nur intern"])], 1),
    ("zwei Konflikte und eine saubere Änderung", _mit(z1="Importer und Kennzahlen.", z3=A3, z7="uv run pytest -q"),
     _mit(z1="Importer für beide Werke.", z5="Lieferungen: daten/roh", z7=A7),
     [B[0], *_k(["Importer und Kennzahlen."], ["Importer für beide Werke."]), B[2], A3, B[4], "Lieferungen: daten/roh", B[6], *_k(["uv run pytest -q"], [A7])], 2),
]
zeige(lambda: "\n".join(drei_wege(B, FAELLE_MERGE[4][1], FAELLE_MERGE[4][2])[0][:7]))


def _t4():
    for name, unsere, ihre, soll, n in FAELLE_MERGE:
        kopien = (list(B), list(unsere), list(ihre))
        ergebnis = drei_wege(B, unsere, ihre)
        assert (list(B), list(unsere), list(ihre)) == kopien, f"{name}: drei_wege hat eine der Eingaben verändert"
        assert isinstance(ergebnis, tuple) and len(ergebnis) == 2, f"{name}: erwartet (zeilen, konflikte), erhalten {ergebnis!r}"
        zeilen, konflikte = ergebnis
        if name.startswith("gleiche") and konflikte:
            raise AssertionError(f"{name}: {konflikte} Konflikt(e), aber gleiche Änderungen auf beiden Seiten sind kein Konflikt")
        assert konflikte == n, f"{name}: {konflikte} Konflikte, erwartet {n}"
        assert zeilen == soll, f"{name}: Ergebnis {zeilen}, erwartet {soll}"


check("TODO 4 neun Fälle wie git merge-file", _t4)

Gleiche Änderungen auf beiden Seiten sind kein Konflikt: Beide wollen dasselbe. Direkt benachbarte Zeilen zählen wie bei Git als eine Stelle, auch wenn jede Seite eine andere Zeile geändert hat. Eine Grenze hat dein Verfahren: Git verkleinert einen Konflikt zusätzlich um Zeilen, die am Rand auf beiden Seiten gleich sind, und rechnet die Unterschiede mit einem eigenen Verfahren; bei den Fällen hier kommt genau dasselbe heraus wie bei `git merge-file`. Einen Konflikt löst du, indem du die Marker entfernst und den Text so schreibst, wie er sein soll. Das ist eine fachliche Entscheidung, keine technische.

## B · Erste Tests mit pytest
*Lerneinheit 4 · Skript Abschnitte 4.3 bis 4.8*

Ab hier geht es um die Meldung der Werkleitung Illinois. Lenas Modul `kennzahlen.py` liest `schichten.csv` aus Modul 0 und rechnet je Werk und Kalenderwoche die Ausschussquote. Die Docstrings beschreiben, was jede Funktion tun soll. Ob der Code das auch tut, weiß niemand: Es gibt keine Tests. Die schreibst du jetzt, eine Funktion nach der anderen.

### Das Notebook als Paket
Deine Tests importieren das Modul so, wie sie es später im Repository tun: `from nordwerk import kennzahlen as kz`. Die nächste Zelle spielt dafür das Paket `nordwerk`: Sie meldet `nordwerk.kennzahlen` als Modul an, das jeden Namen beim Zugriff aus dem Notebook holt. Sie trägt keine Markierung und kommt nicht ins Repository. Ist `nordwerk` in deiner `.venv` schon installiert, verdeckt die Zelle es für diese Sitzung des Kernels.

In [ ]:
# Nur im Notebook: das Paket nordwerk spielen. Nicht ins Repository übernehmen.
import sys
import types


def _aus_dem_notebook(name):
    try:
        return globals()[name]
    except KeyError:
        raise AttributeError(name) from None


_paket = types.ModuleType("nordwerk")
_paket.__path__ = []
_NB_KENNZAHLEN = types.ModuleType("nordwerk.kennzahlen")
_NB_KENNZAHLEN.__getattr__ = _aus_dem_notebook
_paket.kennzahlen = _NB_KENNZAHLEN
sys.modules["nordwerk"] = _paket
sys.modules["nordwerk.kennzahlen"] = _NB_KENNZAHLEN
print("Paket nordwerk im Notebook bereit:", sorted(name for name in sys.modules if name.startswith("nordwerk")))

### Das Modul der Kollegin
Die nächste Zelle ist Lenas Modul, so wie sie es geschickt hat. Sie beginnt mit `# → src/nordwerk/kennzahlen.py`: Am Ende wandert sie in dein Repository. Lies die Docstrings genau, sie sind die Spezifikation. **Ändere die Zelle noch nicht.** Erst in TODO 11 kommst du hierher zurück und behebst, was deine Tests gefunden haben.

In [ ]:
# → src/nordwerk/kennzahlen.py
"""Kennzahlen aus schichten.csv: Ausschussquote, Kalenderwoche, Wochenbericht je Werk.
(Stand der Kollegin, noch ohne Tests)

Liest die Ausgabe von `nordwerk import` (Modul 0) und rechnet daraus Kennzahlen
für den Wochenbericht der Werkleitungen.
"""

import csv
import datetime as dt
import logging
import os
from collections.abc import Iterable, Mapping
from dataclasses import dataclass
from pathlib import Path

log = logging.getLogger("nordwerk.kennzahlen")

GRENZE_VOREINSTELLUNG = 3.0  # Prozent Ausschuss je Woche, ab dem eine Woche auffällig ist
BERICHT_SPALTEN = [
    "werk",
    "kw",
    "schichten",
    "ist_stueck",
    "ausschuss_stueck",
    "quote",
    "hallentemp_c",
]


@dataclass(frozen=True)
class Messung:
    """Eine Zeile aus schichten.csv, reduziert auf das, was die Kennzahlen brauchen."""

    werk: str
    datum: dt.date
    linie: str
    schicht: str
    ist: int
    ausschuss: int
    hallentemp_c: float | None


def lade_schichten(pfad: Path) -> list[Messung]:
    """Liest schichten.csv (UTF-8, Komma); eine leere Temperatur wird zu None."""
    with pfad.open(encoding="utf-8", newline="") as datei:
        return [
            Messung(
                werk=z["werk"],
                datum=dt.date.fromisoformat(z["datum"]),
                linie=z["linie"],
                schicht=z["schicht"],
                ist=int(z["ist_stueck"]),
                ausschuss=int(z["ausschuss_stueck"]),
                hallentemp_c=float(z["hallentemp_c"]) if z["hallentemp_c"] else None,
            )
            for z in csv.DictReader(datei)
        ]


def ausschussquote(ist: int, ausschuss: int) -> float | None:
    """Ausschuss in Prozent der Ist-Stückzahl; None, wenn nichts gefertigt wurde."""
    return 100 * ausschuss / ist


def kalenderwoche(datum: dt.date) -> str:
    """Kalenderwoche nach ISO 8601 als Text, etwa „2026-W27“ (Jahr der Woche, Woche zweistellig)."""
    return datum.strftime("%Y-W%W")


def mittlere_temperatur(messungen: Iterable[Messung]) -> float | None:
    """Mittel der gemessenen Hallentemperaturen.

    Fehlende Werte zählen nicht mit; ohne einen Messwert ist das Ergebnis None.
    """
    liste = list(messungen)
    if not liste:
        return None
    return sum(m.hallentemp_c or 0.0 for m in liste) / len(liste)


def wochenbericht(messungen: Iterable[Messung]) -> list[dict[str, object]]:
    """Je Werk und Kalenderwoche eine Zeile, sortiert nach Werk und Woche.

    Zahl der Schichten, Summen von Ist und Ausschuss, Quote aus den Summen (zwei
    Nachkommastellen) und mittlere Temperatur (eine Nachkommastelle).
    """
    gruppen: dict[tuple[str, str], list[Messung]] = {}
    for m in messungen:
        gruppen.setdefault((m.werk, kalenderwoche(m.datum)), []).append(m)
    bericht: list[dict[str, object]] = []
    for (werk, kw), gruppe in sorted(gruppen.items()):
        ist = sum(m.ist for m in gruppe)
        ausschuss = sum(m.ausschuss for m in gruppe)
        quoten = [ausschussquote(m.ist, m.ausschuss) or 0.0 for m in gruppe]
        quote = sum(quoten) / len(quoten)
        temp = mittlere_temperatur(gruppe)
        bericht.append(
            {
                "werk": werk,
                "kw": kw,
                "schichten": len(gruppe),
                "ist_stueck": ist,
                "ausschuss_stueck": ausschuss,
                "quote": None if quote is None else round(quote, 2),
                "hallentemp_c": None if temp is None else round(temp, 1),
            }
        )
    return bericht


def grenze_aus_umgebung(umgebung: Mapping[str, str] = os.environ) -> float:
    """Grenze für auffällige Wochen in Prozent aus NORDWERK_GRENZE_AUSSCHUSS.

    Fehlt die Variable oder ist sie leer, gilt 3,0. Dezimalpunkt und -komma sind erlaubt.
    Kein Zahlwert oder nicht positiv: ValueError.
    """
    text = umgebung.get("NORDWERK_GRENZE_AUSSCHUSS")
    if text is None or not text.strip():
        return GRENZE_VOREINSTELLUNG
    wert = float(text.strip().replace(",", "."))
    if wert <= 0:
        raise ValueError(f"NORDWERK_GRENZE_AUSSCHUSS muss positiv sein, nicht {text!r}")
    return wert


def auffaellige_wochen(
    bericht: Iterable[dict[str, object]], grenze: float
) -> list[dict[str, object]]:
    """Wochen mit einer Quote über der Grenze, je Woche mit einer Warnung im Log."""
    treffer = []
    for zeile in bericht:
        quote = zeile["quote"]
        if isinstance(quote, float) and quote > grenze:
            log.warning(
                "%s %s: Ausschuss %.2f %% über %.2f %%", zeile["werk"], zeile["kw"], quote, grenze
            )
            treffer.append(zeile)
    return treffer


def schreibe_bericht(bericht: Iterable[dict[str, object]], pfad: Path) -> None:
    """Schreibt den Wochenbericht als CSV (UTF-8, Komma, Zeilenende \\n, leer für None)."""
    pfad.parent.mkdir(parents=True, exist_ok=True)
    with pfad.open("w", encoding="utf-8", newline="") as datei:
        schreiber = csv.DictWriter(datei, fieldnames=BERICHT_SPALTEN, lineterminator="\n")
        schreiber.writeheader()
        for zeile in bericht:
            schreiber.writerow({k: "" if zeile[k] is None else zeile[k] for k in BERICHT_SPALTEN})

In [ ]:
# Der Wochenbericht aus schichten.csv, so wie ihn die Werkleitungen bekommen
with ruhig():
    BERICHT = wochenbericht(lade_schichten(SCHICHTEN_CSV))
    AUFFAELLIG = auffaellige_wochen(BERICHT, grenze_aus_umgebung())
print(len(BERICHT), "Zeilen, erste:", BERICHT[0])
print("auffällig bei", GRENZE_VOREINSTELLUNG, "%:", [(z["werk"], z["kw"], z["quote"]) for z in AUFFAELLIG])

Mit Lenas Fassung steht in der ersten Zeile `2026-W26` mit 2,91 %, und nur `2026-W35` gilt als auffällig. Die erste Lieferung ist aber vom Montag, dem 29. Juni 2026. Genau das meldet die Werkleitung Illinois. Ob die Werkleitung recht hat, findest du mit Tests heraus.

### Tests im Notebook
Auf dem Laptop schreibst du Tests in `tests/test_kennzahlen.py` und startest `uv run pytest`. Hier schreibst du jeden Test in eine Zelle, die mit `# → tests/test_kennzahlen.py` beginnt, und führst sie aus. Die vorgegebene Zelle unten merkt sich jede dieser Zellen. `teste("TODO 5")` schreibt sie dann als `tests/test_kennzahlen.py` in einen frischen Projektordner, zusammen mit einer `pyproject.toml`, die pytest genauso einstellt wie später dein Repository, und startet echtes pytest (`pytest.main`). Du siehst die Ausgabe von pytest, wie im Terminal, nur mit den Tests aus diesem TODO.

Darunter prüft der Selbsttest, ob deine Tests etwas taugen. Er lässt sie in einem einzigen pytest-Lauf gegen mehrere Fassungen des Moduls laufen: gegen die richtige Fassung (dort müssen alle grün sein) und gegen Fassungen mit je einem absichtlich eingebauten Fehler, sogenannte **Mutanten** (Skript Abschnitt 5.7). Ein guter Test wird bei mindestens einem Mutanten rot. Die Fassungen stehen verpackt in der Zelle „Prüfdaten“. Pack sie nicht aus: Der Selbsttest hilft dir nur, wenn du die Fehler mit deinen eigenen Tests findest.

Zwei Dinge zur Arbeitsweise:
- Ein roter Test gegen Lenas Fassung ist ab jetzt ein **Erfolg**: Er zeigt einen Fehler. Behoben wird erst in TODO 11.
- Schreib je TODO eine Testzelle. Änderst du sie, führ sie neu aus und dann ihren Selbsttest. Benennst du alle Tests einer Zelle um und es passt danach nichts mehr zusammen, starte den Kernel neu und führ das Notebook von oben aus.

In [ ]:
# Hilfen für pytest im Notebook: einfach ausführen, nicht ändern.
import ast
import base64
import shutil
import tempfile
import traceback
import zlib

try:
    import pytest
except ModuleNotFoundError:
    raise ModuleNotFoundError("pytest fehlt in diesem Kernel: auf dem Laptop `uv add --dev pytest`, in Colab `!pip install pytest`") from None

MARKE_TESTS = "# → tests/test_kennzahlen.py"
PYPROJECT_TESTS = (
    "[tool.pytest.ini_options]\n"
    'testpaths = ["tests"]\n'
    'addopts = ["-ra", "--strict-markers"]\n'
    'markers = ["daten: braucht die echten Lieferungen in daten/"]\n'
)
PYTEST_OPTIONEN = ["-q", "-p", "no:cacheprovider"]
TEST_TODOS = ["TODO 5", "TODO 6", "TODO 7", "TODO 8", "TODO 9", "TODO 10", "TODO 12", "Bonus 1"]

# 1. Testzellen merken. Python meldet jedes Übersetzen von Code als Ereignis "compile" an alle Audit-Hooks
#    (sys.addaudithook). Dieser Hook merkt sich den Quelltext jeder Zelle, die mit MARKE_TESTS beginnt.
_tz = sys.modules.setdefault("_nordwerk_testzellen", types.ModuleType("_nordwerk_testzellen"))
if not hasattr(_tz, "zellen"):
    _tz.zellen, _tz.letzte = [], None


def _neue_testzelle(quelle):
    try:
        baum = ast.parse(quelle)
    except SyntaxError:
        return
    defs = [k for k in baum.body if isinstance(k, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef))]
    namen = {k.name for k in defs}
    tests = [k.name for k in defs if k.name.startswith("Test" if isinstance(k, ast.ClassDef) else "test")]
    gleich = [z for z in _tz.zellen if (namen & z["namen"]) or z["quelle"] == quelle]
    if gleich:  # dieselbe Zelle noch einmal ausgeführt: alte Fassung ersetzen
        zelle = max(gleich, key=lambda z: len(namen & z["namen"]))
        for z in gleich:
            if z is not zelle and z["label"] in (None, zelle["label"]):
                _tz.zellen.remove(z)
        zelle.update(quelle=quelle, namen=namen, tests=tests)
    else:
        zelle = {"quelle": quelle, "namen": namen, "tests": tests, "label": None}
        _tz.zellen.append(zelle)
    _tz.letzte = zelle


def _beim_uebersetzen(ereignis, argumente):
    if ereignis != "compile" or not argumente:
        return
    quelle = argumente[0]
    if isinstance(quelle, bytes):
        if not quelle.startswith(MARKE_TESTS.encode("utf-8")):
            return
        quelle = quelle.decode("utf-8", "replace")
    if isinstance(quelle, str) and quelle.startswith(MARKE_TESTS):
        _neue_testzelle(quelle)


_tz.melde = _beim_uebersetzen
if not getattr(_tz, "hook", False):
    def _hook(ereignis, argumente):
        try:
            _tz.melde(ereignis, argumente)
        except Exception:
            pass
    sys.addaudithook(_hook)
    _tz.hook = True


def _zellen(label, zuordnen=True):
    """Die Testzellen zu label. Die zuletzt ausgeführte Testzelle gehört zum ersten Selbsttest, der danach läuft."""
    letzte = _tz.letzte
    if zuordnen and letzte is not None and letzte["label"] in (None, label):
        alte = [z for z in _tz.zellen if z["label"] == label and z is not letzte]
        if alte:  # neue Fassung der Zelle mit anderen Namen: ersetzt die alte an ihrer Stelle
            _tz.zellen.remove(letzte)
            _tz.zellen[_tz.zellen.index(alte[0])] = letzte
            for z in alte[1:]:
                _tz.zellen.remove(z)
        letzte["label"] = label
    if zuordnen:
        _tz.letzte = None
    return [z for z in _tz.zellen if z["label"] == label]


def _testdatei():
    """tests/test_kennzahlen.py: die Testzellen ohne Markierung in der Reihenfolge der TODOs; dazu die Zeilen je Zelle."""
    rang = {label: i for i, label in enumerate(TEST_TODOS)}
    teile, bereiche, zeile = [], [], 1
    for z in sorted(_tz.zellen, key=lambda z: rang.get(z["label"], len(rang))):
        text = z["quelle"].split("\n", 1)[1].strip("\n") if "\n" in z["quelle"] else ""
        if text:
            bereiche.append((zeile, zeile + text.count("\n"), z))
            teile.append(text)
            zeile += text.count("\n") + 3
    return "\n\n\n".join(teile) + "\n", bereiche


def zeige_testdatei():
    """Zeigt tests/test_kennzahlen.py so, wie die Datei aus deinen Testzellen entsteht."""
    print(_testdatei()[0])


def _setze_kennzahlen(modul):
    sys.modules["nordwerk.kennzahlen"] = modul
    sys.modules["nordwerk"].kennzahlen = modul


class _Beobachter:
    """pytest-Plugin: tauscht vor dem Import jeder Testdatei das Modul nordwerk.kennzahlen und sammelt die Ergebnisse."""

    def __init__(self, fassungen, rueckwaerts=()):
        self.fassungen, self.rueckwaerts = fassungen, set(rueckwaerts)
        self.ergebnis, self.meldung, self.marken, self.sammelfehler = {}, {}, {}, []

    @pytest.hookimpl(wrapper=True)
    def pytest_make_collect_report(self, collector):
        if isinstance(collector, pytest.Module) and collector.path.name in self.fassungen:
            _setze_kennzahlen(self.fassungen[collector.path.name])
        return (yield)

    def pytest_collectreport(self, report):
        if report.failed:
            self.sammelfehler.append(str(report.longrepr).strip().splitlines()[-1])

    def pytest_collection_modifyitems(self, items):
        if self.rueckwaerts:
            items[:] = [i for i in items if i.path.name not in self.rueckwaerts] + [i for i in reversed(items) if i.path.name in self.rueckwaerts]
        for item in items:
            self.marken[item.nodeid] = {m.name for m in item.iter_markers()}

    def pytest_runtest_logreport(self, report):
        if report.failed:
            self.ergebnis[report.nodeid] = "failed"
            krach = getattr(report.longrepr, "reprcrash", None)
            self.meldung.setdefault(report.nodeid, getattr(krach, "message", None) or str(report.longrepr).strip().splitlines()[-1])
        elif report.skipped:
            self.ergebnis[report.nodeid] = "skipped"
            self.meldung[report.nodeid] = report.longrepr[2] if isinstance(report.longrepr, tuple) else str(report.longrepr)
        elif report.when == "call":
            self.ergebnis.setdefault(report.nodeid, "passed")

    def von(self, datei):
        return {k.split("::", 1)[1]: v for k, v in self.ergebnis.items() if k.startswith(f"tests/{datei}::")}


def _lauf(fassungen, auswahl=None, daten=True, rueckwaerts=(), zeigen=False):
    """Ein pytest-Lauf in einem frischen Projektordner: je Fassung eine Kopie der Testdatei, ausgewählt die Tests in auswahl."""
    text = _testdatei()[0]
    ordner = Path(tempfile.mkdtemp(prefix="nordwerk_pytest_"))
    projekt = ordner / "projekt"
    (projekt / "tests").mkdir(parents=True)
    (projekt / "pyproject.toml").write_text(PYPROJECT_TESTS, encoding="utf-8")
    if daten and SCHICHTEN_CSV.is_file():
        (projekt / "daten").mkdir()
        shutil.copyfile(SCHICHTEN_CSV, projekt / "daten" / "schichten.csv")
    for datei in fassungen:
        (projekt / "tests" / datei).write_text(text, encoding="utf-8")
    vorher = {p.relative_to(projekt).as_posix() for p in projekt.rglob("*")}
    beobachter = _Beobachter(fassungen, rueckwaerts)
    ziele = ["tests"] if auswahl is None else [f"tests/{datei}::{name}" for datei in fassungen for name in auswahl]
    argumente = [*ziele, *PYTEST_OPTIONEN, f"--basetemp={ordner / 'tmp'}"]
    arbeitsordner, pfade, bytecode = os.getcwd(), sys.path[:], sys.dont_write_bytecode
    try:
        os.chdir(projekt)
        sys.dont_write_bytecode = True
        if zeigen:
            beobachter.code = int(pytest.main(argumente, plugins=[beobachter]))
        else:
            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                beobachter.code = int(pytest.main(argumente, plugins=[beobachter]))
        beobachter.neu = {p.relative_to(projekt).as_posix() for p in projekt.rglob("*")} - vorher
    finally:
        os.chdir(arbeitsordner)
        sys.path[:] = pfade
        sys.dont_write_bytecode = bytecode
        for name, modul in list(sys.modules.items()):
            if str(getattr(modul, "__file__", None) or "").startswith(str(ordner)):
                del sys.modules[name]
        _setze_kennzahlen(_NB_KENNZAHLEN)
        shutil.rmtree(ordner, ignore_errors=True)
    beobachter.neu = {n for n in beobachter.neu if "__pycache__" not in n}
    return beobachter


def teste(label=None):
    """Startet pytest mit deinen Tests aus label (ohne label: mit allen) gegen kennzahlen.py im Notebook."""
    zellen = _zellen(label) if label else list(_tz.zellen)
    tests = [t for z in zellen for t in z["tests"]]
    if not tests:
        print("(Noch keine Tests: Führ zuerst die Zelle mit „# → tests/test_kennzahlen.py“ darüber aus.)")
        return
    print(f"pytest mit {'den Tests aus ' + label if label else 'allen Tests'} gegen kennzahlen.py im Notebook:")
    _lauf({"test_kennzahlen.py": _NB_KENNZAHLEN}, tests if label else None, zeigen=True)


def _fassungen(daten):
    """Die richtige Fassung und die Prüffälle als Module, dazu der Plan der Selbsttests."""
    paket = json.loads(zlib.decompress(base64.b64decode(daten)).decode("utf-8"))
    module = {}
    for name in ["richtig", *paket["faelle"]]:
        quelle = paket["richtig"]
        for alt, neu in paket["faelle"][name]["ersetze"] if name != "richtig" else []:
            quelle = quelle.replace(alt, neu)
        modul = types.ModuleType("nordwerk.kennzahlen")
        _setze_kennzahlen(modul)
        try:
            exec(compile(quelle, "<prüfdaten>", "exec"), modul.__dict__)
        finally:
            _setze_kennzahlen(_NB_KENNZAHLEN)
        module[name] = modul
    return module, {name: f["kurz"] for name, f in paket["faelle"].items()}, paket["plan"]


print("pytest", pytest.__version__, "bereit; Testzellen bisher:", len(_tz.zellen))

In [ ]:
# Prüfdaten für die Selbsttests ab TODO 5: die richtige Fassung von kennzahlen.py, Prüffälle mit eingebauten Fehlern
# und der Plan, welcher Prüffall zu welchem TODO gehört. Bitte nicht entpacken: Die Selbsttests helfen dir nur,
# wenn du die Fehler selbst findest.
PRUEFDATEN = (
    "eNrtWutu28gVfpWpCgPkQmZlJ+t4lXrR7Ma5bBwntZ0YjSUoY2kkTUQO1eEwSuU1sA9S9Nc+Rv7lTfZJes4ZDi8ipbhpUqTA"
    "OoAtDs+cOZfv3Ea5amk5nBo5aXVZq0f/ngillnwaCsV4mrBkOEUKoYJh8rbL7qUJrMCvv6exEW32hAPhSOhFPJzC4zn+UZeC"
    "uLI3gp0LPQt6yjs1XI0YELIncRiKiVRtpoCYxVMl2JlITOL3VE8dSfjIRlLgSRN+KdjbWLHXKtajBbBiMprH2rxm3tN4lIas"
    "47MU+GoxBDawj2uUuVChp8Yf3ms4V62IhpKgbKGQJlUTUA9OtxaATfYUBhrnn0fcCCMjwXjCRiZfDuMJ6DLJn+MEjtRxxIao"
    "5tDIWCUBvxxmgrPHRmh+GYKpnvL5nHYSObDnw5AniUhYcaJdykjm3ExDeeleP4dHlBUkYAdOjmAizBF8FNrrtZzRgllhjxZZ"
    "+eHJ4fGrw8HLZyeHj49Pzw6Pjl4cPwQut4IOY39kz3W8FMoUziZPWg/zS7BdxIRU2RKgZDz+8GsYygmTCVjmh8OTxz8+Ohuc"
    "Pr93dHZ4DGwveorBT6+F0vRabfc4W5QecqCV1oDfIDGpGJZ3cSdV/RWBsvQ85eAEZUQ0HwxpuY/a99Rfctt6Y9L14EynAkxD"
    "a+ypSBIARdexwX+HqPArIUNRD4s2AHCULqUAt4A1wHNJmy0QKIDjUjxdAjoRhEGBNDwArdJlidH2EWRLoy6ALEDM2bVQKilK"
    "NNnxpRUwVZdJZexTbqPSWtkYXTYOY27Yz+w4VsIaZSTGLOQjMch18+ZjPuoS0ny2/T1IkZiLzDj9inVs2FaMwrwXZw+29yFF"
    "xFHE/bsWMqEQGuM9mkMcmFSzhdQjtkxJjtwuZBVppgwFCOI5iCLUMB4Bwg96rdSMt/fBnUyJBRhGHOAmnwITDCYzwfBHCzhC"
    "Ofy5n0wDr7rqPHGwvHBA7bfrJOSdg8w5AQamTOJxrCNuPNxJ72Gr37CXvEj86VPzAZkRiSz73EwIzjgA59Kp5UBpPDoHRL6l"
    "HkaNG8uoOSDQ0O5qZPV9JsesYZ2JMBHk2xXWfvURDMiWAFVMucF9OTQnApCoPXJoibZfQJVXKpHn8N9ewT4Btwz2Cm6LDAdn"
    "u7SHteFxYrZPzYf3wxnG7l2rAuBDKabQJQmbiDHEu5wYtkj1qIpdMAbIww4OWKeORhtxpYWdTod9U4jN/mTzqFN0Vi6xXjU7"
    "kHaQAypKVWoyUxyq7OPTZ2x/r7PDeJhA9L0DMwmz4Oy3X/6129nd2z7fvfPbL/9k3k98qm1tLNVztlwIkEhgivcrer4BcrCK"
    "pR1ApifpAoiIIcnAtedXVB33Wle46Xr7/Iq2dTu7o2ubCq22kTQmhBwxMHmO8CIbsEJ18/qZp6HN/n0K3ERIGk0EshEK8vCj"
    "HKT2AFv9adsDMSXjYXNgQPMPv1LmJp+jbHdtw4K5TFEiWWDSR2dD0meHeiIulUyybFYSJM/0wBQKYhSUA4XgHyEEc00RQitE"
    "wFbFhlj3c5jhCnG9Ac6SNPKI1geIAd/sobD9otwifdzqVA5GEKwXgMA2iy/fQMPTrxaGn2wLSF1aFZciL6htlkBDQ9WTwJpv"
    "IPTlrnkFgUiePHVlps1O0ygCY2GPCBFLm/KQbrO/Ymagco0NYEbrIZotx2M4bYbVicAtlG0mHQLLVcojaVfo/QYPT3Q6n6PJ"
    "yC4mnYPRyDrwq9+uVtA+IOHq2u6rI6Dk0IxpkAgDbuJpaDwvCrBGtVeyQxRQBPp+m130/YDjtpEXZUGYeba7znOIzH4hj5ed"
    "sABuVgQUEH0lRp6TSQI6E8/3S+JS6iO0YSowhW52TymdFznP0Rcr63dRwocd9QpQSv4leowfIG/KLDXemYmc5apV6qpeHl2r"
    "0KXeod1EgF1uF6zY+LLU9XYpJjOJGonLVb6LZm6kqhf2bmGWxh1Z29yllIFZxVo4y2O2husYYsPLpr7dZvmqxb9gRw5o4Ibr"
    "bbazyuy6eKxWj8w5RcKaQOpeigFoN0gjSL3Y1bkPXTdhFfEHIIiTQKi3UseqKByVhPWQWDIaG4vJJpt1VLlNwLxy/Ozk/vnh"
    "yZNBNlHde3F6+uMj+FUpKHagfcm1xCTKYkxiGBiJtA0xhJcEolvtTsDui6WMeDhP1czms23KOEALn4UOeXppsrzzBJISZUWq"
    "QcTV1ql5nEgjYVp/ycNUHGod64ZUZaANAIs4e+HoCEPjWo1odMyqDm11DoU4xSqEawFYWc49v16MGifOoiiCHLa3LHMJtJiH"
    "fCg8HN2g3e+1oPnwCylo459XeiwuE1HS2xtvUIlFmGgya7FE0KUEWfAK5fiDvs6VztTAI8st6HjMqS0SA1s7vZVEmxfOerJt"
    "Z+jNRrGeunFNzaAI+YyKqM7q3If3cCwVSIvhdj61l0jPuVbgbCYjdhRPKs2c0WIMLe1KEVjSyCvz4OvW8zCRXORppF+qBNgH"
    "S5UYrsCNWfIgdWFcAzxbDt87S1TTQAjyLUBc2Tiq9VpbCdtKSjdSbCvYHbOtrcwQ2RMCxwnoprp8AVMzPGaCWSk2zimZjVx5"
    "ID5VhGQUBUhAOC3kpRi4vupm4FgZvWud7allaxrutrDH//H0ZXUAb9teS1Fz24MfwDrmHpvpkH21tafJe87BJCaIZiOpPfuQ"
    "0FUJjA/vsBrFM3dz0jCyg8HR+p80uTurIRzdTHiuodnIZkKAkRThSHHolw5WbpywzVICSCOpuIk1ntJTeShX2AcLZDq102aJ"
    "4CPQb+Ch44V3NeuSe2gaJozN+tXCl6/iATNkviL8NQgBBmpRYhFQRK9a83Fnhz4IDR3gEhcvLlqfbTDH027GDFN9x2+3EJ+t"
    "WaqXIIlKw/AaHkHI3bqQn8MgKN+VvSDzHLW/gXyNdLfq0n18BsezS1Qr72EH7AOqNUfebj7yM07NK/JZRmCoMV5VQwBu/W37"
    "fOucoI+Jzp7/9V99OLW+qKQfESE311c6tDsbYceA4uHfYmb3K4cRzQ0PW1EAQz7oFMoSKyeLfSgb6z+Zz5wG+R6FVl7ZRSNk"
    "m5UmQ78mk52Z+vXOBPWxnJ3E2dNKzLawP6erH438EFsPOFmy8pVVK4vrb5vj+ve4ulFcVewOPi/s/rrq+9fO3ntfVx7doEDl"
    "NiaX/05d/htPO9QL1Ij9dSVnv9lUv+ewG+SwDX5tuDvKvfvdxsbixphDQou7fwjeRN7s8Z3Ol+26/ls+7Gd2tXpz5cavhq+g"
    "2msusNyWyhdd63q9nZ1ml/y/FqcqFiu39A6FO7vrVa4P02Xl3FupxrG3HmZr2uf/4TUMCr1OutufLwYrhBsjr6EN+GK3Eigo"
    "Mv80zlVWa9TZW4+gz3aPUwbeBqabOFRazQ13MqsYz6+114i5idUaizXU9U+4fUZBN2/D+yGvtLe8uSYayRZyRbKdPbv/jFmc"
    "jiU0Jrh6YVtY3HcpErz8yFdv0Sqcms4gHmgVxNHxO8i18IK47TVw22u5UujCscZnzjWPBHQvCMCc2Z06M8g0rmtyVq4xo/SS"
    "DCJuhlOkAgeAbUp89xuE3G+54lDjN5bv8GtgeqGxuCw4JLUE3gGVyLl+18B1t+XKsJW6gbuBJgT/8xa9ifVICb3KOaviVUM4"
    "1t+2XIDWWA/5HIKjpHhW+prdkzEqW6LYuNuw8XbLNeBrTYe3ccpqNlViYB8L7X6IVZqwnY2YWeXZv76+/jcTJw9F"
)
FASSUNG, KURZTEXT, PLAN_TESTS = _fassungen(PRUEFDATEN)
print(len(FASSUNG) - 1, "Prüffälle geladen")

In [ ]:
# Selbsttests der Testaufgaben: einfach ausführen, nicht ändern. Welcher Prüffall zu welchem TODO gehört, steht in PLAN_TESTS
# aus den Prüfdaten.


def _funktionen(baeume):
    return [k for b in baeume for k in b.body if isinstance(k, (ast.FunctionDef, ast.AsyncFunctionDef))]


def _leer(funktion):
    rumpf = [k for k in funktion.body if not (isinstance(k, ast.Expr) and isinstance(k.value, ast.Constant))]
    return all(isinstance(k, ast.Pass) for k in rumpf)


def _ist_fixture(funktion):
    return any("fixture" in ast.unparse(d) for d in funktion.decorator_list)


def _aufrufe(baeume, name):
    return [k for b in baeume for k in ast.walk(b) if isinstance(k, ast.Call)
            and (getattr(k.func, "attr", None) == name or getattr(k.func, "id", None) == name)]


def _nutzt(baeume, name):
    return any((isinstance(k, ast.Attribute) and k.attr == name) or (isinstance(k, ast.Name) and k.id == name)
               for b in baeume for k in ast.walk(b))


def _parameter(baeume, name):
    return any(name in [a.arg for a in f.args.args] for f in _funktionen(baeume) if f.name.startswith("test"))


def _raises_mit_match(baeume):
    aufrufe = _aufrufe(baeume, "raises")
    if not aufrufe:
        return "kein `pytest.raises` für die ungültigen Werte"
    if not all(any(w.arg == "match" for w in k.keywords) for k in aufrufe):
        return "`pytest.raises` ohne `match`: So wäre der Test auch grün, wenn der `ValueError` aus einem ganz anderen Grund käme"
    return True


STRUKTUR = {
    "approx": lambda b: _nutzt(b, "approx") or "kein `pytest.approx`: Kommazahlen vergleichst du mit Toleranz",
    "parametrize": lambda b: _nutzt(b, "parametrize") or "kein `@pytest.mark.parametrize`: Schreib die Grenzwerte als Tabelle von Fällen",
    "raises_match": _raises_mit_match,
    "setenv": lambda b: bool(_aufrufe(b, "setenv")) or "kein `monkeypatch.setenv`: Setz die Variable einmal so und ruf die Funktion ohne Argument auf",
    "fixture": lambda b: (any(f.name == "messungen_klein" and _ist_fixture(f) for f in _funktionen(b)) and _parameter(b, "messungen_klein"))
    or "keine Fixture `messungen_klein` mit `@pytest.fixture`, die deine Tests als Parameter bekommen",
    "tmp_path": lambda b: _parameter(b, "tmp_path") or "kein Test bekommt `tmp_path`: Dateien gehören in den Ordner, den pytest je Test anlegt",
    "caplog": lambda b: _parameter(b, "caplog") or "kein Test bekommt `caplog`",
    "daten": lambda b: True,
}


def _ladeprobe(label):
    """Lädt die ganze Testdatei einmal mit der richtigen Fassung; ein Fehler beim Laden gehört zu einer Zelle."""
    text, bereiche = _testdatei()
    try:
        _setze_kennzahlen(FASSUNG["richtig"])
        exec(compile(text, "tests/test_kennzahlen.py", "exec"), {"__name__": "_ladeprobe"})
    except Exception as e:
        if isinstance(e, SyntaxError):
            nr = e.lineno or 0
        else:
            nr = max([f.lineno for f in traceback.extract_tb(e.__traceback__) if f.filename == "tests/test_kennzahlen.py"] or [0])
        zelle = next((z for a, b, z in bereiche if a <= nr <= b), None)
        if zelle is not None and zelle["label"] not in (None, label):
            raise NotImplementedError(zelle["label"]) from None
        raise AssertionError(f"die Testdatei lässt sich nicht laden ({type(e).__name__}: {e})") from None
    finally:
        _setze_kennzahlen(_NB_KENNZAHLEN)


def _vorpruefung(label):
    """Gemeinsamer Anfang: Testzelle da, Importe da, nicht leer, Fixtures aus anderen TODOs fertig, Datei ladbar."""
    zellen = _zellen(label)
    if not zellen:
        raise AssertionError("noch keine Testzelle ausgeführt: Führ zuerst die Zelle mit „# → tests/test_kennzahlen.py“ darüber aus")
    if label != "TODO 5" and not _zellen("TODO 5", zuordnen=False):
        raise NotImplementedError("TODO 5")  # dort stehen die Importe der Testdatei
    baeume = [ast.parse(z["quelle"]) for z in zellen]
    if all(_leer(f) for f in _funktionen(baeume)):
        raise NotImplementedError(label)
    fremde = {z["label"]: z for z in _tz.zellen if z["label"] not in (None, label)}
    for anderes, z in fremde.items():
        fixtures = {f.name for f in _funktionen([ast.parse(z["quelle"])]) if _ist_fixture(f)}
        if any(_parameter(baeume, name) for name in fixtures):
            braucht(anderes)
    tests = [t for z in zellen for t in z["tests"]]
    assert tests, "keine Testfunktion gefunden: Namen von Testfunktionen beginnen mit test_"
    _ladeprobe(label)
    return tests, baeume


def _alle_gruen(beob, datei, wo):
    ergebnis = beob.von(datei)
    assert ergebnis, f"pytest hat {wo} keinen deiner Tests ausgeführt" + (f" ({beob.sammelfehler[0]})" if beob.sammelfehler else "")
    for name, art in ergebnis.items():
        meldung = beob.meldung.get(f"tests/{datei}::{name}", "")
        if art == "skipped":
            raise AssertionError(f"{wo} wird {name} übersprungen: {meldung}")
        if art != "passed":
            raise AssertionError(f"{wo} scheitert {name}: {meldung}")


def pruefe_tests(label):
    """Selbsttest einer Testaufgabe: richtige Fassung grün, jeder Prüffall des TODOs rot, Aufbau wie verlangt."""
    plan = PLAN_TESTS[label]
    tests, baeume = _vorpruefung(label)
    fassungen = {"test_richtig.py": FASSUNG["richtig"]}
    fassungen.update({f"test_{name}.py": FASSUNG[name] for name in plan["finden"] + plan.get("bestehen", [])})
    rueck = ["test_rueckwaerts.py"] if plan.get("rueckwaerts") else []
    if rueck:
        fassungen["test_rueckwaerts.py"] = FASSUNG["richtig"]
    beob = _lauf(fassungen, tests, rueckwaerts=rueck)
    if beob.sammelfehler:
        raise AssertionError(f"pytest kann die Testdatei nicht sammeln: {beob.sammelfehler[0]}")
    if rueck:
        vorwaerts = {n for n, a in beob.von("test_richtig.py").items() if a != "passed"}
        rueckwaerts = {n for n, a in beob.von("test_rueckwaerts.py").items() if a != "passed"}
        if vorwaerts != rueckwaerts:
            raise AssertionError(f"das Ergebnis hängt von der Reihenfolge der Tests ab ({sorted(vorwaerts ^ rueckwaerts)[0]}): "
                                 "Teilen sich deine Tests eine Liste? Eine Fixture soll jedem Test frische Daten geben")
    _alle_gruen(beob, "test_richtig.py", "an der richtigen Fassung")
    if plan.get("ordner") and beob.neu:
        raise AssertionError(f"dein Test hat {sorted(beob.neu)[-1]} im Projektordner angelegt: Schreib Dateien nach tmp_path")
    for name in plan.get("bestehen", []):
        if any(a != "passed" for a in beob.von(f"test_{name}.py").values()):
            raise AssertionError("deine Tests lehnen eine gleichwertige Rechnung mit anderen Rundungsfehlern ab: Vergleich Kommazahlen mit `pytest.approx`")
    gefunden = [n for n in plan["finden"] if any(a == "failed" for a in beob.von(f"test_{n}.py").values())]
    if len(gefunden) < len(plan["finden"]):
        raise AssertionError(f"Deine Tests finden noch nicht alle Fehler, die zu diesem TODO gehören ({len(gefunden)} von {len(plan['finden'])} gefunden)")
    for regel in plan["struktur"]:
        ok = STRUKTUR[regel](baeume)
        if ok is not True:
            raise AssertionError(ok)
    if plan.get("ohne_daten"):
        if not all("daten" in beob.marken.get(f"tests/test_richtig.py::{n}", set()) for n in beob.von("test_richtig.py")):
            raise AssertionError("der Test trägt die Marke `@pytest.mark.daten` nicht")
        ohne = _lauf({"test_richtig.py": FASSUNG["richtig"]}, tests, daten=False)
        for name, art in ohne.von("test_richtig.py").items():
            grund = ohne.meldung.get(f"tests/test_richtig.py::{name}", "")
            assert art == "skipped", f"ohne daten/schichten.csv {'scheitert' if art == 'failed' else 'läuft'} {name}, statt sich zu überspringen: {grund}"
            grund = grund.split(":", 1)[1] if grund.startswith("Skipped") and ":" in grund else grund
            assert grund.strip() and grund.strip() not in ("Skipped", "unconditional skip"), "beim Überspringen fehlt eine Begründung (reason)"
    benannt = [KURZTEXT[n] for n in plan["finden"] if KURZTEXT.get(n)]
    weitere = len(plan["finden"]) - len(benannt)
    if benannt:
        text = ", ".join(benannt[:-1]) + (" und " if len(benannt) > 1 else "") + benannt[-1]
        if weitere:
            text += " und " + ("einen weiteren eingebauten Fehler" if weitere == 1 else f"{weitere} weitere eingebaute Fehler")
    else:
        text = f"alle {weitere} eingebauten Fehler"
    return f"deine Tests bestehen die richtige Fassung und finden {text}" + (", ohne Daten überspringt er sich" if plan.get("ohne_daten") else "")


print("Selbsttests für", ", ".join(label for label in PLAN_TESTS if label != "TODO 11"), "bereit")

### TODO 5 · Erste Tests für `ausschussquote`
Das Skript schreibt erste Tests für `lies_temperatur` aus dem Importer und liest eine rote Ausgabe von pytest Zeile für Zeile. Hier testest du Lenas `ausschussquote(ist, ausschuss)`.

**Ziel:** zwei Tests, die das Verhalten aus dem Docstring prüfen: einen normalen Wert und den Fall ohne gefertigte Stücke.

**Regeln**
- Die Zelle enthält schon die Importe der ganzen Testdatei (`import datetime as dt`, `import pytest`, `from nordwerk import kennzahlen as kz` und zwei weitere für spätere TODOs). Ruf alles über `kz.` auf, etwa `kz.ausschussquote(2016, 61)`.
- Ein Test ist eine Funktion, deren Name mit `test_` beginnt. Darin rufst du die Funktion auf und prüfst das Ergebnis mit `assert`.
- 2016 Stück mit 61 Ausschuss sind etwa 3,0258 %. Kommazahlen vergleichst du nie mit `==` auf die letzte Stelle, sondern mit `pytest.approx`, etwa `== pytest.approx(3.0258, abs=1e-4)`.
- Ohne gefertigte Stücke (`ist == 0`) verlangt der Docstring `None`. Prüf das mit `is None`.

In [ ]:
# → tests/test_kennzahlen.py
"""Tests für nordwerk.kennzahlen: Quote, Kalenderwoche, Grenze, Wochenbericht, Dateien und Logs."""

import csv
import datetime as dt
from pathlib import Path

import pytest

from nordwerk import kennzahlen as kz


def test_ausschussquote_normaler_wert():
    """2016 Stück mit 61 Ausschuss ergeben etwa 3,03 Prozent."""
    ...  # TODO 5: mit pytest.approx vergleichen


def test_ausschussquote_ohne_stueck():
    """Ohne gefertigte Stücke gibt es keine Quote."""
    ...  # TODO 5

In [ ]:
# Selbsttest TODO 5
teste("TODO 5")
check("TODO 5 Tests für ausschussquote", lambda: pruefe_tests("TODO 5"))

Gegen Lenas Fassung ist mindestens ein Test rot. pytest zeigt dir die Zeile in deinem Test, die Zeile im Modul, in der es passiert, und die Art des Fehlers; wie du die Ausgabe liest, zeigt das Skript in Abschnitt 4.4. Der Selbsttest zeigt, dass deine Tests an der richtigen Fassung grün sind und den eingebauten Fehler finden.

### TODO 6 · Grenzwerte mit `parametrize` für `kalenderwoche`
Das Skript zeigt `@pytest.mark.parametrize` an den Grenzwerten von `zeile()` (Abschnitte 4.7 und 4.8). Hier testest du `kalenderwoche(datum)`. Laut Docstring gilt ISO 8601: Die Woche beginnt am Montag, und die erste Woche eines Jahres ist die mit dem ersten Donnerstag. Deshalb gehören Tage um den Jahreswechsel manchmal zum Nachbarjahr.

**Ziel:** ein Test, der viele Fälle als Tabelle prüft.

**Regeln**
- `@pytest.mark.parametrize(("datum", "erwartet"), [(…, …), …])` über einer Testfunktion `test_kalenderwoche(datum, erwartet)`. pytest macht aus jeder Zeile der Tabelle einen eigenen Test.
- Mindestens diese Fälle: `2026-06-29 → 2026-W27` (der Montag der ersten Lieferung), `2024-12-30 → 2025-W01`, `2027-01-03 → 2026-W53`. Ein Datum schreibst du als `dt.date(2026, 6, 29)`.
- Such dir weitere Grenzfälle: ein Sonntag, der 1. Januar eines Jahres.

In [ ]:
# → tests/test_kennzahlen.py
@pytest.mark.parametrize(
    ("datum", "erwartet"),
    [
        (dt.date(2026, 6, 29), "2026-W27"),  # Montag der ersten Lieferwoche
        # TODO 6: weitere Fälle, vor allem um den Jahreswechsel
    ],
)
def test_kalenderwoche(datum, erwartet):
    """Kalenderwoche nach ISO 8601, auch um den Jahreswechsel."""
    ...  # TODO 6

In [ ]:
# Selbsttest TODO 6
teste("TODO 6")
check("TODO 6 Grenzwerte für kalenderwoche", lambda: pruefe_tests("TODO 6"))

Jede Zeile der Tabelle erscheint in pytest als eigener Test mit den Werten in eckigen Klammern. So siehst du auf einen Blick, welche Fälle rot sind. Ein einzelner Wert mitten im Jahr hätte nur einen Teil gezeigt; erst die Grenzfälle am Jahreswechsel zeigen das ganze Bild.

### TODO 7 · Ausnahmen und Umgebung für `grenze_aus_umgebung`
Das Skript prüft eine Ausnahme mit `pytest.raises(…, match=…)` und setzt eine Umgebungsvariable mit `monkeypatch.setenv` (Abschnitte 4.6 und 5.3). Hier testest du `grenze_aus_umgebung(umgebung)`: Sie liest die Grenze für auffällige Wochen aus `NORDWERK_GRENZE_AUSSCHUSS`.

**Ziel:** Tests für die Voreinstellung, das Dezimalkomma, ungültige Werte und das Lesen aus `os.environ`.

**Regeln**
- Ohne Variable (`kz.grenze_aus_umgebung({})`) gilt 3,0; `"2,5"` ergibt 2,5.
- `"abc"` und `"-1"` lösen `ValueError` aus. Prüf das mit `with pytest.raises(ValueError, match=…):`. `match` ist ein regulärer Ausdruck, der in der Meldung vorkommen muss; such dir für jeden Fall ein Stück der Meldung aus, das genau diesen Grund zeigt.
- Einmal ohne Argument: Setz die Variable mit `monkeypatch.setenv("NORDWERK_GRENZE_AUSSCHUSS", …)` und ruf `kz.grenze_aus_umgebung()` auf. `monkeypatch` stellt die Umgebung nach dem Test wieder her.

In [ ]:
# → tests/test_kennzahlen.py
def test_grenze_voreinstellung():
    """Ohne NORDWERK_GRENZE_AUSSCHUSS gilt 3,0."""
    ...  # TODO 7


def test_grenze_mit_komma():
    """„2,5“ ergibt 2,5."""
    ...  # TODO 7


def test_grenze_ungueltig():
    """„abc“ und „-1“ lösen ValueError aus, jeweils mit passender Meldung."""
    ...  # TODO 7


def test_grenze_aus_os_environ(monkeypatch):
    """Ohne Argument liest die Funktion os.environ, und zwar beim Aufruf."""
    ...  # TODO 7

In [ ]:
# Selbsttest TODO 7
teste("TODO 7")
check("TODO 7 Ausnahmen und Umgebung", lambda: pruefe_tests("TODO 7"))

Hier sind auch gegen Lenas Fassung alle Tests grün: Nicht jede Funktion hat einen Fehler. Die Tests sind trotzdem wertvoll. Sie halten fest, was gelten soll, und schlagen an, sobald jemand die Funktion später ändert. Ohne `match` wäre `pytest.raises` zu großzügig: Ein Tippfehler im Code, der irgendeinen `ValueError` auslöst, sähe aus wie die gewollte Prüfung.

## C · Fixtures und Testdaten
*Lerneinheit 5 · Skript Abschnitte 5.1 bis 5.4*

### TODO 8 · Fixture mit kleinen Testdaten
Das Skript baut eine Fixture `stammwerk_werte` für die Tests des Importers (Abschnitt 5.1). Hier baust du kleine, absichtlich gewählte Messungen für `mittlere_temperatur` und `wochenbericht`. Echte Daten haben Hunderte Zeilen; von Hand nachrechnen kannst du nur wenige.

**Ziel:** eine Fixture `messungen_klein` und Tests, die das Verhalten aus den Docstrings an Werten prüfen, die du selbst nachgerechnet hast.

**Regeln**
- `@pytest.fixture` über `def messungen_klein():`, die eine Liste von `kz.Messung(werk, datum, linie, schicht, ist, ausschuss, hallentemp_c)` zurückgibt. Ein Test bekommt sie, indem er einen Parameter mit genau diesem Namen hat. pytest ruft die Fixture für jeden Test neu auf, jeder Test bekommt also frische Daten.
- Bau die Daten so, dass ein Fehler auffällt: mehrere Schichten derselben Woche mit **ungleich großen** Stückzahlen und mindestens eine Schicht **ohne Temperatur** (`None`).
- Test für `mittlere_temperatur`: Fehlende Werte zählen nicht mit, ohne einen einzigen Messwert ist das Ergebnis `None`.
- Test für `wochenbericht`: Rechne die Zeile einer Woche von Hand nach (Schichten, Summen, Quote aus den Summen mit zwei Nachkommastellen, Temperatur mit einer) und vergleiche den ganzen Bericht.

In [ ]:
# → tests/test_kennzahlen.py
@pytest.fixture
def messungen_klein():
    """Kleine Testdaten: mehrere Schichten einer Woche, ungleich groß, eine ohne Temperatur."""
    ...  # TODO 8: eine Liste von kz.Messung zurückgeben


def test_mittlere_temperatur_ohne_fehlende_werte(messungen_klein):
    """Fehlende Temperaturen zählen nicht mit."""
    ...  # TODO 8


def test_wochenbericht_quote_aus_summen(messungen_klein):
    """Quote aus den Summen, von Hand nachgerechnet."""
    ...  # TODO 8

In [ ]:
# Selbsttest TODO 8
teste("TODO 8")
check("TODO 8 Fixture und kleine Testdaten", lambda: pruefe_tests("TODO 8"))

Mit vier Schichten kannst du jede Zahl im Kopf nachrechnen: 8 Stück Ausschuss von 500 sind 1,6 %, und die Temperatur der Woche ist das Mittel der beiden gemessenen Werte. Kleine, absichtlich gebaute Daten zeigen einen Fehler, den du in Hunderten echter Zeilen nie sähest. Der Selbsttest lässt deine Tests auch in umgekehrter Reihenfolge laufen. Liefert die Fixture jedem Test eine eigene Liste, ändert das nichts.

### TODO 9 · Dateien mit `tmp_path`
Das Skript testet `importiere()` an zwei selbst geschriebenen Lieferdateien in `tmp_path` (Abschnitt 5.2). Hier testest du `schreibe_bericht` und `lade_schichten`. `tmp_path` ist ein `pathlib.Path` auf einen leeren Ordner, den pytest für jeden Test neu anlegt. Dein Projektordner bleibt sauber, und kein Test sieht die Dateien eines anderen.

**Ziel:** zwei Tests mit Dateien in `tmp_path`.

**Regeln**
- Schreib einen kleinen Bericht (eine Liste von Wörterbüchern mit den Schlüsseln aus `kz.BERICHT_SPALTEN`, eine Zeile davon mit `hallentemp_c` gleich `None`) mit `kz.schreibe_bericht(bericht, pfad)`, lies die Datei mit `csv.DictReader` wieder ein und vergleiche: gleiche Spalten in derselben Reihenfolge, gleiche Werte als Text, `None` als leeres Feld.
- Schreib eine kleine `schichten.csv` mit Kopfzeile und einer Zeile mit leerer Temperatur, lies sie mit `kz.lade_schichten(pfad)` und vergleiche mit der erwarteten `kz.Messung`.
- Alle Dateien liegen in `tmp_path`, keine im Arbeitsordner.

In [ ]:
# → tests/test_kennzahlen.py
def test_bericht_schreiben_und_wieder_lesen(tmp_path):
    """Ein Bericht mit einer fehlenden Temperatur übersteht Schreiben und Lesen."""
    ...  # TODO 9


def test_lade_schichten_leere_temperatur(tmp_path):
    """Eine leere Temperatur in schichten.csv wird zu None."""
    ...  # TODO 9

In [ ]:
# Selbsttest TODO 9
teste("TODO 9")
check("TODO 9 Dateien mit tmp_path", lambda: pruefe_tests("TODO 9"))

Ein Test, der in einen festen Pfad schreibt, hinterlässt Dateien im Projekt, kann eine echte Datei überschreiben und stört andere Tests, die denselben Pfad nehmen. Mit `tmp_path` passiert nichts davon. Der Selbsttest merkt es, wenn ein Test etwas im Projektordner anlegt.

### TODO 10 · Logs prüfen mit `caplog`
Das Skript prüft mit `caplog` die Warnungen von `zeilen()` und mit `capsys` die Ausgabe von `main()` (Abschnitt 5.4). Hier prüfst du `auffaellige_wochen(bericht, grenze)`: Laut Docstring meldet sie jede Woche über der Grenze mit einer Warnung im Log.

**Ziel:** ein Test, der Rückgabe und Log-Meldungen von `auffaellige_wochen` prüft.

**Regeln**
- Ein kleiner Bericht genügt, etwa drei Zeilen mit `werk`, `kw` und `quote`: zwei über der Grenze 3,0, eine darunter.
- Prüf die Rückgabe: genau die beiden Wochen über der Grenze.
- `caplog.records` ist die Liste aller Meldungen während des Tests. Prüf für die Meldungen des Loggers `nordwerk.kennzahlen` (`r.name`): genau eine je Woche über der Grenze, Stufe `WARNING` (`r.levelname`), und die Meldung (`r.getMessage()`) nennt Werk und KW, etwa `Illinois 2026-W27`.

In [ ]:
# → tests/test_kennzahlen.py
def test_auffaellige_wochen_warnt_je_woche(caplog):
    """Je Woche über der Grenze genau eine WARNING mit Werk und KW, sonst keine."""
    ...  # TODO 10

In [ ]:
# Selbsttest TODO 10
teste("TODO 10")
check("TODO 10 Logs mit caplog", lambda: pruefe_tests("TODO 10"))

Logs sind Teil des Verhaltens: Die Werkleitungen sehen die Warnung, nicht die Liste. Prüfst du nur den Text, aber nicht die Stufe, fiele es nicht auf, wenn die Meldung nur noch als INFO käme und im Betrieb gar nicht mehr erscheint.

## D · Fehler beheben und Integrationstest
*Lerneinheit 5 · Skript Abschnitte 5.5 bis 5.7*

### TODO 11 · Die Fehler beheben
Deine Tests aus TODO 5 bis 10 sind gegen Lenas Fassung an einigen Stellen rot. Jeder rote Test zeigt auf eine Stelle, an der der Code nicht tut, was der Docstring sagt. Jetzt behebst du sie, wie bei einem Regressionstest (Skript Abschnitt 5.6): Der rote Test ist schon da, nach dem Fix wird er grün und bleibt es.

**Ziel:** Die Zelle mit `# → src/nordwerk/kennzahlen.py` in Teil B tut überall, was ihre Docstrings sagen.

**Regeln**
- Geh zur Zelle `# → src/nordwerk/kennzahlen.py` in Teil B, korrigiere sie und führ sie neu aus. Dann führ die Zelle unten aus.
- Ändere nur, was ein roter Test zeigt, und nur so, wie der Docstring es verlangt. Alles andere bleibt, wie Lena es geschrieben hat: Namen, Signaturen, Docstrings.
- `teste()` startet pytest mit allen deinen Tests gegen deine korrigierte Fassung. Der Selbsttest prüft, dass sie grün sind, dass sie jeden der Fehler einzeln finden würden und dass der Wochenbericht aus `schichten.csv` stimmt.

In [ ]:
# Selbsttest TODO 11
teste()
VORHER_TESTS = ["TODO 5", "TODO 6", "TODO 7", "TODO 8", "TODO 9", "TODO 10"]


def _t11_tests():
    braucht(*VORHER_TESTS)
    tests = [t for label in VORHER_TESTS for z in _zellen(label, zuordnen=False) for t in z["tests"]]
    faelle = PLAN_TESTS["TODO 11"]["finden"]
    beob = _lauf({"test_deine.py": _NB_KENNZAHLEN, **{f"test_{name}.py": FASSUNG[name] for name in faelle}}, tests)
    _alle_gruen(beob, "test_deine.py", "an deiner Fassung")
    gefunden = sum(any(a == "failed" for a in beob.von(f"test_{name}.py").values()) for name in faelle)
    assert gefunden == len(faelle), f"deine Tests aus TODO 5 bis 10 finden noch nicht jeden Fehler einzeln ({gefunden} von {len(faelle)} gefunden)"
    return "deine Tests sind an deiner Fassung grün und finden jeden der vier Fehler einzeln"


def _gleich(beschreibung, deine, richtig):
    """Vergleicht ein Ergebnis deiner Fassung mit der richtigen; Ausnahmen zählen als Ergebnis."""
    ergebnisse = []
    for aufruf in (deine, richtig):
        try:
            ergebnisse.append(("Wert", aufruf()))
        except Exception as e:
            ergebnisse.append(("Ausnahme", type(e).__name__))
    (art_d, d), (art_r, r) = ergebnisse
    if art_d == art_r == "Wert" and isinstance(d, list) and isinstance(r, list) and len(d) == len(r):
        for i, (x, y) in enumerate(zip(d, r)):
            assert x == y, f"{beschreibung}: Eintrag {i} ist {x!r:.200}, laut Docstring {y!r:.200}"
        return
    gleich = art_d == art_r and (d == r if not (isinstance(d, float) and isinstance(r, float)) else abs(d - r) < 1e-9)
    assert gleich, f"{beschreibung} ergibt {d!r:.200}{' (Ausnahme)' if art_d == 'Ausnahme' else ''}, laut Docstring {r!r:.200}{' (Ausnahme)' if art_r == 'Ausnahme' else ''}"


def _rufe(modul, name, args):
    """Ruft modul.name(*args) auf; schreibe_bericht liefert die geschriebenen Bytes, Messungen werden zu Tupeln."""
    if name == "schreibe_bericht":
        with tempfile.TemporaryDirectory() as ordner:
            pfad = Path(ordner) / "bericht.csv"
            getattr(modul, name)(*args, pfad)
            return pfad.read_bytes()
    with ruhig():
        wert = getattr(modul, name)(*args)
    if isinstance(wert, list):
        return [tuple(vars(w).values()) if hasattr(w, "__dataclass_fields__") else w for w in wert]
    return wert


def _t11_modul():
    braucht(*VORHER_TESTS)
    kz, ref = _NB_KENNZAHLEN, FASSUNG["richtig"]
    # jede öffentliche Funktion des Moduls an denselben Proben wie die richtige Fassung
    meine, ihre = kz.lade_schichten(SCHICHTEN_CSV), ref.lade_schichten(SCHICHTEN_CSV)
    soll = ref.wochenbericht(ihre)
    tage = [dt.date(2024, 1, 1) + dt.timedelta(days=n) for n in range(1461)]
    umgebungen = [{}, {"NORDWERK_GRENZE_AUSSCHUSS": "2,5"}, {"NORDWERK_GRENZE_AUSSCHUSS": "abc"}, {"NORDWERK_GRENZE_AUSSCHUSS": "0"}]
    proben = [("lade_schichten", "daten/schichten.csv", (SCHICHTEN_CSV,), (SCHICHTEN_CSV,))]
    proben += [("ausschussquote", f"{i}, {a}", (i, a), (i, a)) for i in (0, 1, 7, 500, 2016) for a in (0, 1, 8, 61) if a <= i]
    proben += [("kalenderwoche", str(t), (t,), (t,)) for t in tage]
    proben += [("mittlere_temperatur", f"Zeilen {a} bis {b} aus schichten.csv", (meine[a:b],), (ihre[a:b],)) for a, b in [(0, 0), (0, 229), (229, 892), (0, 892)]]
    proben += [("wochenbericht", "alle Zeilen aus schichten.csv", (meine,), (ihre,))]
    proben += [("grenze_aus_umgebung", repr(u), (u,), (u,)) for u in umgebungen]
    proben += [("auffaellige_wochen", f"Wochenbericht, {g}", (soll, g), (soll, g)) for g in (2.5, 3.0)]
    proben += [("schreibe_bericht", "Wochenbericht", (soll,), (soll,))]
    for name, text, args_d, args_r in proben:
        _gleich(f"{name}({text})", lambda: _rufe(kz, name, args_d), lambda: _rufe(ref, name, args_r))
    bericht = kz.wochenbericht(meine)
    assert len(bericht) == 23, f"wochenbericht hat {len(bericht)} Zeilen, erwartet 23 (Illinois 12, Stammwerk 11 Wochen)"
    assert (sum(z["ist_stueck"] for z in bericht), sum(z["ausschuss_stueck"] for z in bericht)) == (165737, 3789), "Summen von Ist und Ausschuss stimmen nicht"
    with ruhig():
        treffer = [(z["werk"], z["kw"], z["quote"]) for z in kz.auffaellige_wochen(bericht, kz.GRENZE_VOREINSTELLUNG)]
    assert treffer == [("Illinois", "2026-W27", 3.03), ("Illinois", "2026-W36", 3.1)], f"auffällig sind {treffer}"
    return "der Wochenbericht hat 23 Zeilen, auffällig sind Illinois 2026-W27 und 2026-W36"


check("TODO 11 deine Tests gegen deine Fassung", _t11_tests)
check("TODO 11 Wochenbericht wie im Docstring", _t11_modul)

Mit den Korrekturen beginnt der Bericht mit `2026-W27` und einer Quote von 3,03 %. Über der Grenze von 3,0 % liegen genau zwei Wochen aus Illinois: KW 27 mit 3,03 % und KW 36 mit 3,10 %. Die Werkleitung hatte recht. Jeder deiner roten Tests von vorher ist jetzt ein Regressionstest: Kommt einer der Fehler je zurück, wird er wieder rot.

### TODO 12 · Integrationstest an echten Daten (Meldung aus Illinois)
Bisher hat jeder Test eine Funktion für sich geprüft (Unit-Tests). Ein **Integrationstest** prüft das Zusammenspiel an echten Daten: `lade_schichten`, `wochenbericht` und `auffaellige_wochen` hintereinander auf `schichten.csv`. Das Skript zeigt die Marke `daten` und das Überspringen am Importer (Abschnitt 5.5).

**Ziel:** ein Test, der die Antwort auf die Meldung aus Illinois festhält. Machst du die Schritte auf deinem Laptop der Reihe nach, heißt die Meldung in deinem Repository Issue #3.

**Regeln**
- Der Test liest `daten/schichten.csv` (`SCHICHTEN` in der Zelle) und prüft: Über der Grenze `kz.GRENZE_VOREINSTELLUNG` liegen genau `("Illinois", "2026-W27")` und `("Illinois", "2026-W36")`, in dieser Reihenfolge.
- Er trägt die Marke `@pytest.mark.daten`. Sie ist in der `pyproject.toml` des Notebooks registriert, wie später in deinem Repository; mit `--strict-markers` wäre ein Tippfehler in der Marke ein Fehler.
- Fehlt die Datei, überspringt er sich mit klarer Begründung: `@pytest.mark.skipif(not SCHICHTEN.is_file(), reason="…")` oder `pytest.skip("…")` im Test.

In [ ]:
# → tests/test_kennzahlen.py
SCHICHTEN = Path("daten") / "schichten.csv"


def test_meldung_illinois_auffaellige_wochen():
    """Meldung aus Illinois: Welche Wochen liegen in den echten Daten über der Grenze?"""
    ...  # TODO 12: Marke daten, ohne Datei überspringen

In [ ]:
# Selbsttest TODO 12
teste("TODO 12")
check("TODO 12 Integrationstest für die Meldung aus Illinois", lambda: pruefe_tests("TODO 12"))

Der Test hält die Antwort auf die Meldung aus Illinois fest: Er ist gegen Lenas Fassung rot und gegen die korrigierte grün. Im Repository läuft er überall, wo `daten/schichten.csv` liegt; fehlt die Datei, steht er mit Begründung unter „skipped“, statt rot zu werden. In der Testpyramide stehen viele schnelle Unit-Tests unten und wenige Integrationstests darüber (Skript Abschnitt 4.9).

## Bonus
### Bonus 1 · Eigenschaftstest ohne Bibliothek
Das Skript prüft an `lies_temperatur` eine Eigenschaft statt einzelner Fälle (Abschnitt 4.8). Hier prüfst du `kalenderwoche` an **jedem** Tag von 2024 bis 2027, ohne für jeden Tag das Ergebnis zu kennen.

**Regeln**
- Geh mit `dt.timedelta(days=1)` vom 1. Januar 2024 bis zum 31. Dezember 2027.
- Jeder Tag hat dieselbe Kalenderwoche wie der Montag seiner Woche (`tag - dt.timedelta(days=tag.weekday())`).
- Die Nummer hinter `-W` ist zweistellig und liegt zwischen 01 und 53.
- Das Jahr vor `-W` ist das Jahr des Donnerstags derselben Woche (Montag plus drei Tage): So legt ISO 8601 den Jahreswechsel fest.

In [ ]:
# → tests/test_kennzahlen.py
def test_kalenderwoche_eigenschaften():
    """Für jeden Tag 2024 bis 2027: KW wie am Montag, Nummer 01 bis 53, Jahr des Donnerstags."""
    ...  # Bonus 1

In [ ]:
# Selbsttest Bonus 1
teste("Bonus 1")
check("Bonus 1 Eigenschaften der Kalenderwoche", lambda: pruefe_tests("Bonus 1"))

1.461 Tage, drei Eigenschaften, kein einziges von Hand ausgerechnetes Ergebnis. Eigenschaftstests finden Fälle, an die beim Aufschreiben einzelner Beispiele niemand denkt. Bibliotheken wie Hypothesis erzeugen die Eingaben dafür sogar zufällig und suchen gezielt nach Gegenbeispielen.

### Bonus 2 · Commit-Nachrichten prüfen
Das Skript nennt die Regeln für gute Commit-Nachrichten (Abschnitt 1.5). Eine Funktion, die sie prüft, kann später als `commit-msg`-Hook jeden Commit vor dem Speichern prüfen (Zusatz im Schritt „Auf deinem Laptop“).

**Ziel:** `pruefe_commit_nachricht(text)` liefert die Liste der verletzten Regeln, leer heißt in Ordnung.

**Regeln**
- Zeilen, die mit `#` beginnen, überspringst du (so sehen die Hinweise aus, die Git in die Vorlage für die Nachricht schreibt). Leerzeichen am Zeilenende zählen nicht. Windows-Zeilenenden `\r\n` gehen auch.
- Ist die erste Zeile (der Betreff) leer, liefere nur `["leer"]`.
- Sonst in dieser Reihenfolge: `"zu_lang"`, wenn der Betreff mehr als 72 Zeichen hat; `"punkt"`, wenn er mit einem Punkt endet; `"leerzeile"`, wenn nach dem Betreff noch Text folgt, die zweite Zeile aber nicht leer ist.

In [ ]:
# Bonus 2
def pruefe_commit_nachricht(text):
    """Die verletzten Regeln einer Commit-Nachricht: leer, zu_lang, punkt, leerzeile."""
    raise NotImplementedError("Bonus 2")


zeige(lambda: pruefe_commit_nachricht("Fehler behoben."))

In [ ]:
# Selbsttest Bonus 2
def _b2():
    for kuerzel, nachricht in NACHRICHTEN.items():
        assert pruefe_commit_nachricht(nachricht) == [], f"die Nachricht von {kuerzel} ist in Ordnung, erhalten {pruefe_commit_nachricht(nachricht)}"
    faelle = [
        ("", ["leer"]), ("   \n\nText", ["leer"]), ("Fehler behoben.", ["punkt"]), ("x" * 72, []), ("x" * 73, ["zu_lang"]),
        ("Kennzahlen getestet\nQuote aus den Summen", ["leerzeile"]), ("Kennzahlen getestet.\nQuote", ["punkt", "leerzeile"]),
        ("# Bitte Nachricht eingeben\nIllinois: Schichten erkennen\n\nFixes #1\n", []),
        ("Illinois: Schichten erkennen   \r\n\r\nFixes #1\r\n", []), ("y" * 80 + ".", ["zu_lang", "punkt"]),
    ]
    for text, soll in faelle:
        ist = pruefe_commit_nachricht(text)
        assert ist == soll, f"pruefe_commit_nachricht({text[:30]!r}) ergibt {ist}, erwartet {soll}"


check("Bonus 2 Commit-Nachrichten", _b2)

Als Hook bekommt das Programm den Pfad einer Datei mit der Nachricht. Es liest sie, ruft deine Funktion auf und beendet sich mit einem Exit-Code ungleich 0, wenn die Liste nicht leer ist. Dann bricht Git den Commit ab, und die Nachricht lässt sich verbessern.

## Ins Repo
Jetzt wandern Modul und Tests in dein Repository. Wie, steht im Schritt **„Auf deinem Laptop“** der Kursseite (Schritt L10, Branch `feature/kennzahlen` und Pull Request). Übernimm jede Zelle, die mit `# → …` beginnt, ohne diese erste Zeile und in der Reihenfolge des Notebooks, in die genannte Datei. Zwischen zwei Zellen lässt du zwei Leerzeilen.

| Zellen im Notebook | Datei im Repository |
|---|---|
| das Modul der Kollegin in Teil B, mit deinen Korrekturen aus TODO 11 | `src/nordwerk/kennzahlen.py` |
| die Testzellen aus TODO 5 bis 10 und TODO 12, Bonus 1, wenn du ihn gelöst hast | `tests/test_kennzahlen.py` |

Die Zelle unten zeigt `tests/test_kennzahlen.py` genau so, wie sie aus deinen Testzellen entsteht. Die Marke `daten` registrierst du in `pyproject.toml` unter `[tool.pytest.ini_options]`, wie im Schritt „Auf deinem Laptop“ beschrieben. Damit der Integrationstest dort läuft statt sich zu überspringen, muss `schichten.csv` unter `daten/` liegen: `uv run nordwerk import --aus daten` schreibt sie mit der Prüfsumme `0bc618f824be` dorthin (Schritt L10). Dazu kommt die mitgelieferte `tests/conftest.py`: Ihre autouse-Fixture setzt den Logger `nordwerk` nach jedem Test zurück, damit dein `caplog`-Test aus TODO 10 auch nach Tests mit `main()` alle Meldungen sieht (Skript Abschnitt 5.1). Nicht ins Repository gehören die Hilfen, die Datenzelle, die Zelle „Prüfdaten“, die Selbsttests, die Übungen aus Teil A und Bonus 2.

**Nachweis:** `uv run pytest` ist in deinem Repository grün, und dein Pull Request zeigt in der CI ein grünes Häkchen.

In [ ]:
# So entsteht tests/test_kennzahlen.py aus deinen Testzellen
zeige_testdatei()

## Zum Nachdenken

1. Deine Tests aus TODO 7 waren schon gegen Lenas Fassung grün. Waren sie damit überflüssig?
2. Der Selbsttest hat deine Tests gegen Fassungen mit eingebauten Fehlern laufen lassen. Woran erkennst du auf deinem Laptop, ohne diese Fassungen, ob deine Tests etwas taugen?
3. Kim und du ändert in zwei Branches dieselbe Zeile in `pyproject.toml`. Wer entscheidet beim Konflikt, welche Fassung gilt, und wo hältst du die Entscheidung fest?

<details><summary>Hinweise zu den Antworten</summary>

1. Nein. Ein Test, der jetzt grün ist, hält fest, was gelten soll. Wird `grenze_aus_umgebung` später geändert, etwa damit sie auch Prozentzeichen versteht, merkt dein Test sofort, wenn dabei das Dezimalkomma verloren geht. Tests sind ein Sicherheitsnetz für künftige Änderungen, nicht nur ein Werkzeug zur Fehlersuche (Skript Abschnitt 4.1).
2. Mach es selbst: Bau absichtlich einen kleinen Fehler ein, etwa `>` statt `>=`, und schau, ob ein Test rot wird. Bleibt alles grün, fehlt ein Test. Werkzeuge für Mutationstests machen das automatisch; die Testabdeckung zeigt dagegen nur, welche Zeilen gelaufen sind, nicht, ob ein Test das Ergebnis geprüft hat (Skript Abschnitt 5.7).
3. Die Technik zeigt nur, dass beide Seiten dieselbe Stelle geändert haben. Welche Fassung richtig ist, ist eine fachliche Frage; ihr klärt sie zu zweit, im Zweifel im Pull Request. Die Entscheidung steht dann in der Commit-Nachricht des Merges und im Review.
</details>

## Wie es weitergeht

Du hast Inhalte über ihre Git-ID erkannt, eine Historie mit Branches durchsucht, die Merge-Basis bestimmt, „vorne“ und „hinten“ gezählt und zwei Seiten mit einem Dreiwege-Merge zusammengeführt. Für Lenas Modul hast du Tests mit `approx`, `parametrize`, `raises`, `monkeypatch`, Fixtures, `tmp_path` und `caplog` geschrieben, mit ihnen vier Fehler gefunden und behoben und die Meldung aus Illinois mit einem Integrationstest beantwortet. Auf deinem Laptop laufen dieselben Tests mit pre-commit und in der CI bei jedem Pull Request. Ab Modul 2 bauen beide Kurse auf diesem Repository auf: Advanced Data Science mit Modul 2 „Große Daten und Datenverträge“, Advanced AI Engineering mit Modul 2 „Training in PyTorch“.